# Загрузка данных

In [ ]:
# pip install pandas numpy matplotlib requests yt-dlp ffmpeg-python Pillow scikit-learn

In [9]:
import ast
import json
import re
import tarfile
import tempfile
from collections import Counter
from pathlib import Path
from typing import List, Set

import pandas as pd
import requests
from tqdm import tqdm


# =========================
# CONFIG
# =========================

MTG_OUT_DIR = Path("MTG_Jamendo_50")
MTG_AUDIO_DIR = MTG_OUT_DIR / "audio"
MTG_META_OUT = MTG_OUT_DIR / "mtg_jamendo_50_metadata.csv"

N_GENRES = 10
TRACKS_PER_GENRE = 6

# Используем valid.tsv, как в твоём текущем коде.
# Если хочешь train, поменяй на:
# MTG_METADATA_SPLIT = "train"
# MTG_AUDIO_SPLIT_DIR = "train"
MTG_METADATA_SPLIT = "valid"
MTG_AUDIO_SPLIT_DIR = "val"

MAX_SHARDS_TO_SCAN = 200

HF_REPO_BASE = "https://huggingface.co/datasets/rkstgr/mtg-jamendo/resolve/main"

MTG_TSV_URL = f"{HF_REPO_BASE}/{MTG_METADATA_SPLIT}.tsv"
MTG_TAR_URL_TEMPLATE = f"{HF_REPO_BASE}/data/{MTG_AUDIO_SPLIT_DIR}/{{shard_id}}.tar"

REQUIRED_OUTPUT_COLUMNS = [
    "id",
    "artist_id",
    "album_id",
    "durationInSec",
    "selected_genre",
    "genres_list",
    "instruments_list",
    "moods_list",
    "audio_downloaded",
    "audio_path",
]


# =========================
# UTILS
# =========================

def safe_filename(text: str, max_len: int = 80) -> str:
    text = str(text).strip()
    text = re.sub(r"[^\w\-а-яА-ЯёЁ]+", "_", text, flags=re.UNICODE)
    text = re.sub(r"_+", "_", text)
    return text[:max_len].strip("_") or "untitled"


def normalize_tag(tag: str) -> str:
    tag = str(tag).strip().lower()
    tag = tag.replace("-", "_")
    tag = re.sub(r"\s+", "_", tag)
    tag = re.sub(r"_+", "_", tag)
    return tag.strip("_")


def parse_list_cell(value) -> List[str]:
    """
    Преобразует ячейку из TSV/CSV в список тегов.

    Поддерживает:
    - "['rock', 'pop']"
    - '["rock", "pop"]'
    - "rock,pop"
    - пустые значения
    """
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return [normalize_tag(x) for x in value if str(x).strip()]

    value = str(value).strip()

    if not value or value.lower() in {"nan", "none", "null", "[]"}:
        return []

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [normalize_tag(x) for x in parsed if str(x).strip()]
    except Exception:
        pass

    return [normalize_tag(x) for x in re.split(r"[,;|]", value) if x.strip()]


def serialize_list(tags: List[str]) -> str:
    """
    Сохраняем списки тегов в CSV как JSON-строки.
    Это удобнее и стабильнее, чем Python repr.
    """
    return json.dumps(tags, ensure_ascii=False)


def download_file(url: str, out_path: Path, chunk_size: int = 1024 * 1024) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))

        with open(out_path, "wb") as f, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=out_path.name,
            leave=False,
        ) as pbar:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))


def is_nonempty_list(value) -> bool:
    return len(parse_list_cell(value)) > 0


def file_exists_and_nonempty(path: str) -> bool:
    if not path:
        return False

    p = Path(path)
    return p.exists() and p.is_file() and p.stat().st_size > 0


# =========================
# LOAD AND FILTER METADATA
# =========================

def load_mtg_metadata() -> pd.DataFrame:
    print(f"Downloading MTG-Jamendo metadata from: {MTG_TSV_URL}")

    df = pd.read_csv(MTG_TSV_URL, sep="\t")

    required_columns = {
        "id",
        "artist_id",
        "album_id",
        "durationInSec",
        "genres",
        "instruments",
        "moods",
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(
            f"В {MTG_METADATA_SPLIT}.tsv не найдены нужные столбцы: {missing}. "
            f"Есть столбцы: {list(df.columns)}"
        )

    df = df.copy()

    df["genres_list"] = df["genres"].apply(parse_list_cell)
    df["instruments_list"] = df["instruments"].apply(parse_list_cell)
    df["moods_list"] = df["moods"].apply(parse_list_cell)

    df["durationInSec"] = pd.to_numeric(df["durationInSec"], errors="coerce")

    before = len(df)

    # Самая важная фильтрация:
    # оставляем только треки, где есть все три вида разметки.
    df = df[
        df["id"].notna()
        & df["artist_id"].notna()
        & df["album_id"].notna()
        & df["durationInSec"].notna()
        & (df["durationInSec"] > 0)
        & df["genres_list"].apply(lambda x: len(x) > 0)
        & df["instruments_list"].apply(lambda x: len(x) > 0)
        & df["moods_list"].apply(lambda x: len(x) > 0)
    ].copy()

    after = len(df)

    print(f"Metadata filtering: {before} -> {after}")
    print("Kept only tracks with non-empty genres, instruments and moods.")

    if df.empty:
        raise ValueError("После фильтрации не осталось треков с полной разметкой.")

    return df


def get_top_genres_from_complete_metadata(df: pd.DataFrame, n_genres: int) -> List[str]:
    counter = Counter()

    for genres in df["genres_list"]:
        counter.update(genres)

    if not counter:
        raise ValueError("Не удалось посчитать жанры: genres_list пустой.")

    top_genres = [genre for genre, _ in counter.most_common(n_genres)]

    print("\nTop genres among tracks with complete metadata:")
    for genre, count in counter.most_common(n_genres):
        print(f"  {genre}: {count}")

    return top_genres


def select_tracks_by_top_genres(
    df: pd.DataFrame,
    top_genres: List[str],
    tracks_per_genre: int,
) -> pd.DataFrame:
    """
    Выбирает по tracks_per_genre уникальных треков для каждого популярного жанра.

    Важно:
    - кандидаты уже отфильтрованы по полной разметке;
    - один track_id не используется дважды;
    - если для жанра не хватает треков, код падает с ошибкой, а не добирает плохие примеры.
    """
    selected_rows = []
    used_ids: Set[int] = set()

    for genre in top_genres:
        candidates = df[
            df["genres_list"].apply(lambda tags: genre in tags)
        ].copy()

        candidates = candidates.sort_values(
            by=["durationInSec", "id"],
            ascending=[False, True],
        )

        genre_selected = 0

        for _, row in candidates.iterrows():
            track_id = int(row["id"])

            if track_id in used_ids:
                continue

            row_dict = row.to_dict()
            row_dict["selected_genre"] = genre

            selected_rows.append(row_dict)
            used_ids.add(track_id)
            genre_selected += 1

            if genre_selected >= tracks_per_genre:
                break

        if genre_selected < tracks_per_genre:
            raise ValueError(
                f"Для жанра '{genre}' найдено только {genre_selected} треков "
                f"с полной разметкой, а нужно {tracks_per_genre}. "
                f"Уменьши TRACKS_PER_GENRE или выбери другой split."
            )

    selected = pd.DataFrame(selected_rows)

    expected = len(top_genres) * tracks_per_genre

    if len(selected) != expected:
        raise ValueError(f"Ожидалось {expected} треков, выбрано {len(selected)}.")

    return selected


# =========================
# DOWNLOAD AUDIO
# =========================

def extract_selected_tracks_from_hf_tars(
    selected: pd.DataFrame,
    max_shards: int,
) -> pd.DataFrame:
    """
    Скачивает tar-шарды по одному и извлекает только нужные .opus.

    Если аудио уже есть на диске, оно не скачивается повторно.
    """
    MTG_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

    selected = selected.copy()

    target_ids: Set[int] = set(selected["id"].astype(int).tolist())
    found_ids: Set[int] = set()

    id_to_selected_genre = {
        int(row["id"]): str(row["selected_genre"])
        for _, row in selected.iterrows()
    }

    # Сначала проверяем уже скачанные файлы.
    for track_id in list(target_ids):
        selected_genre = safe_filename(id_to_selected_genre[track_id])
        expected_path = MTG_AUDIO_DIR / selected_genre / f"{track_id}.opus"

        if expected_path.exists() and expected_path.stat().st_size > 0:
            found_ids.add(track_id)

    print(f"\nNeed tracks: {len(target_ids)}")
    print(f"Already on disk: {len(found_ids)}")
    print(f"Remaining to extract: {len(target_ids - found_ids)}")

    for shard_id in range(max_shards):
        remaining = target_ids - found_ids

        if not remaining:
            break

        url = MTG_TAR_URL_TEMPLATE.format(shard_id=shard_id)

        with tempfile.TemporaryDirectory() as tmpdir:
            tar_path = Path(tmpdir) / f"{shard_id}.tar"

            print(f"\nDownloading shard {shard_id}. Remaining tracks: {len(remaining)}")
            print(f"URL: {url}")

            try:
                download_file(url, tar_path)
            except Exception as e:
                print(f"Failed to download shard {shard_id}: {e}")
                continue

            try:
                with tarfile.open(tar_path, "r") as tar:
                    for member in tar.getmembers():
                        if not member.isfile():
                            continue

                        basename = Path(member.name).name
                        suffix = Path(basename).suffix.lower()
                        stem = Path(basename).stem

                        if suffix != ".opus":
                            continue

                        try:
                            track_id = int(stem)
                        except ValueError:
                            continue

                        if track_id not in remaining:
                            continue

                        selected_genre = safe_filename(id_to_selected_genre[track_id])
                        genre_dir = MTG_AUDIO_DIR / selected_genre
                        genre_dir.mkdir(parents=True, exist_ok=True)

                        out_path = genre_dir / f"{track_id}.opus"

                        extracted = tar.extractfile(member)

                        if extracted is None:
                            continue

                        with open(out_path, "wb") as f:
                            f.write(extracted.read())

                        if out_path.exists() and out_path.stat().st_size > 0:
                            found_ids.add(track_id)
                            print(f"Extracted: {track_id} -> {out_path}")

            except Exception as e:
                print(f"Failed to process shard {shard_id}: {e}")

    selected["audio_downloaded"] = selected["id"].astype(int).apply(
        lambda track_id: track_id in found_ids
    )

    selected["audio_path"] = selected.apply(
        lambda row: str(
            MTG_AUDIO_DIR
            / safe_filename(row["selected_genre"])
            / f"{int(row['id'])}.opus"
        )
        if int(row["id"]) in found_ids
        else "",
        axis=1,
    )

    missing = target_ids - found_ids

    if missing:
        raise RuntimeError(
            f"Не удалось найти/скачать {len(missing)} треков: "
            f"{sorted(list(missing))[:30]}. "
            f"Увеличь MAX_SHARDS_TO_SCAN или выбери другой split."
        )

    return selected


# =========================
# FINAL VALIDATION
# =========================

def validate_final_metadata(df: pd.DataFrame) -> None:
    """
    Жёсткая проверка итогового CSV.
    Если что-то не так — падаем с понятной ошибкой.
    """
    missing_cols = set(REQUIRED_OUTPUT_COLUMNS) - set(df.columns)

    if missing_cols:
        raise ValueError(f"В итоговой таблице нет столбцов: {missing_cols}")

    if df[REQUIRED_OUTPUT_COLUMNS].isna().any().any():
        bad_cols = df[REQUIRED_OUTPUT_COLUMNS].columns[
            df[REQUIRED_OUTPUT_COLUMNS].isna().any()
        ].tolist()
        raise ValueError(f"В итоговой таблице есть NaN в столбцах: {bad_cols}")

    empty_string_cols = []

    for col in REQUIRED_OUTPUT_COLUMNS:
        if df[col].astype(str).str.strip().eq("").any():
            empty_string_cols.append(col)

    if empty_string_cols:
        raise ValueError(f"В итоговой таблице есть пустые строки в столбцах: {empty_string_cols}")

    for list_col in ["genres_list", "instruments_list", "moods_list"]:
        empty_mask = df[list_col].apply(lambda x: len(parse_list_cell(x)) == 0)

        if empty_mask.any():
            bad_ids = df.loc[empty_mask, "id"].astype(str).tolist()
            raise ValueError(
                f"В столбце {list_col} есть пустые списки. "
                f"Проблемные id: {bad_ids[:20]}"
            )

    if not df["audio_downloaded"].astype(bool).all():
        bad_ids = df.loc[~df["audio_downloaded"].astype(bool), "id"].astype(str).tolist()
        raise ValueError(f"Не все аудио скачаны. Проблемные id: {bad_ids[:20]}")

    bad_audio_paths = df[
        ~df["audio_path"].apply(file_exists_and_nonempty)
    ]["id"].astype(str).tolist()

    if bad_audio_paths:
        raise ValueError(
            f"Для некоторых треков audio_path не существует или файл пустой. "
            f"Проблемные id: {bad_audio_paths[:20]}"
        )

    if len(df) != N_GENRES * TRACKS_PER_GENRE:
        raise ValueError(
            f"Ожидалось {N_GENRES * TRACKS_PER_GENRE} треков, "
            f"а получилось {len(df)}."
        )

    print("\nFinal metadata validation passed.")
    print("No missing values.")
    print("No empty genres_list / instruments_list / moods_list.")
    print("All audio files exist.")


def save_final_metadata(selected: pd.DataFrame) -> pd.DataFrame:
    selected = selected.copy()

    # Сохраняем списки тегов как JSON-строки.
    for col in ["genres_list", "instruments_list", "moods_list"]:
        selected[col] = selected[col].apply(serialize_list)

    final_df = selected[REQUIRED_OUTPUT_COLUMNS].copy()

    validate_final_metadata(final_df)

    MTG_OUT_DIR.mkdir(parents=True, exist_ok=True)
    final_df.to_csv(MTG_META_OUT, index=False, encoding="utf-8-sig")

    print(f"\nSaved final metadata: {MTG_META_OUT}")
    print(f"Tracks: {len(final_df)}")

    print("\nSelected genre distribution:")
    print(final_df["selected_genre"].value_counts())

    print("\nExample rows:")
    print(final_df.head())

    return final_df


# =========================
# MAIN PIPELINE
# =========================

def prepare_mtg_jamendo_audio_subset() -> pd.DataFrame:
    MTG_OUT_DIR.mkdir(parents=True, exist_ok=True)

    metadata = load_mtg_metadata()

    top_genres = get_top_genres_from_complete_metadata(
        df=metadata,
        n_genres=N_GENRES,
    )

    selected = select_tracks_by_top_genres(
        df=metadata,
        top_genres=top_genres,
        tracks_per_genre=TRACKS_PER_GENRE,
    )

    selected = extract_selected_tracks_from_hf_tars(
        selected=selected,
        max_shards=MAX_SHARDS_TO_SCAN,
    )

    final_df = save_final_metadata(selected)

    return final_df


if __name__ == "__main__":
    print("=== Preparing MTG-Jamendo audio subset ===")

    final_metadata = prepare_mtg_jamendo_audio_subset()

    print("\nDone.")

=== Preparing MTG-Jamendo audio subset ===
Metadata filtering: 5719 -> 1319
Kept only tracks with non-empty genres, instruments and moods.

Top genres among tracks with complete metadata:
  soundtrack: 322
  electronic: 299
  ambient: 222
  pop: 207
  classical: 194
  easylistening: 139
  rock: 122
  orchestral: 111
  chillout: 107
  folk: 104

Need tracks: 60
Already on disk: 0
Remaining to extract: 60

URL: https://huggingface.co/datasets/rkstgr/mtg-jamendo/resolve/main/data/val/0.tar



URL: https://huggingface.co/datasets/rkstgr/mtg-jamendo/resolve/main/data/val/1.tar


Extracted: 635794 -> MTG_Jamendo_50\audio\ambient\635794.opus
Extracted: 1352999 -> MTG_Jamendo_50\audio\chillout\1352999.opus
Extracted: 1313461 -> MTG_Jamendo_50\audio\chillout\1313461.opus
Extracted: 1027667 -> MTG_Jamendo_50\audio\easylistening\1027667.opus
Extracted: 1066201 -> MTG_Jamendo_50\audio\easylistening\1066201.opus
Extracted: 1185080 -> MTG_Jamendo_50\audio\ambient\1185080.opus
Extracted: 1116940 -> MTG_Jamendo_50\audio\ambient\1116940.opus
Extracted: 1116941 -> MTG_Jamendo_50\audio\ambient\1116941.opus
Extracted: 1327694 -> MTG_Jamendo_50\audio\chillout\1327694.opus

URL: https://huggingface.co/datasets/rkstgr/mtg-jamendo/resolve/main/data/val/2.tar


Extracted: 759341 -> MTG_Jamendo_50\audio\orchestral\759341.opus
Extracted: 759342 -> MTG_Jamendo_50\audio\orchestral\759342.opus
Extracted: 1028503 -> MTG_Jamendo_50\audio\ambient\1028503.opus
Extracted: 1035759 -> MTG_Jamendo_50\audio\ambient\1035759.opus

URL: https://huggingface.co/datasets/rkstgr/mtg-jamendo/resolve/main/data/val/3.tar


Extracted: 965443 -> MTG_Jamendo_50\audio\classical\965443.opus
Extracted: 1189144 -> MTG_Jamendo_50\audio\easylistening\1189144.opus

URL: https://huggingface.co/datasets/rkstgr/mtg-jamendo/resolve/main/data/val/4.tar


...


URL: https://huggingface.co/datasets/rkstgr/mtg-jamendo/resolve/main/data/val/18.tar


Extracted: 1289382 -> MTG_Jamendo_50\audio\rock\1289382.opus
Extracted: 1147362 -> MTG_Jamendo_50\audio\rock\1147362.opus
Extracted: 1221745 -> MTG_Jamendo_50\audio\rock\1221745.opus
Extracted: 1384753 -> MTG_Jamendo_50\audio\rock\1384753.opus

URL: https://huggingface.co/datasets/rkstgr/mtg-jamendo/resolve/main/data/val/19.tar


Extracted: 1123782 -> MTG_Jamendo_50\audio\rock\1123782.opus

URL: https://huggingface.co/datasets/rkstgr/mtg-jamendo/resolve/main/data/val/20.tar


Extracted: 1095111 -> MTG_Jamendo_50\audio\chillout\1095111.opus
Extracted: 1062598 -> MTG_Jamendo_50\audio\easylistening\1062598.opus
Extracted: 1073223 -> MTG_Jamendo_50\audio\easylistening\1073223.opus
Extracted: 794804 -> MTG_Jamendo_50\audio\orchestral\794804.opus
Extracted: 1171147 -> MTG_Jamendo_50\audio\classical\1171147.opus
Extracted: 973615 -> MTG_Jamendo_50\audio\orchestral\973615.opus
Extracted: 874646 -> MTG_Jamendo_50\audio\soundtrack\874646.opus
Extracted: 1068617 -> MTG_Jamendo_50\audio\soundtrack\1068617.opus

URL: https://huggingface.co/datasets/rkstgr/mtg-jamendo/resolve/main/data/val/21.tar


Extracted: 1047453 -> MTG_Jamendo_50\audio\classical\1047453.opus
Extracted: 1295654 -> MTG_Jamendo_50\audio\pop\1295654.opus
Extracted: 858721 -> MTG_Jamendo_50\audio\classical\858721.opus
Extracted: 1027388 -> MTG_Jamendo_50\audio\soundtrack\1027388.opus
Extracted: 778561 -> MTG_Jamendo_50\audio\soundtrack\778561.opus
Extracted: 800960 -> MTG_Jamendo_50\audio\chillout\800960.opus
Extracted: 1059974 -> MTG_Jamendo_50\audio\classical\1059974.opus
Extracted: 961897 -> MTG_Jamendo_50\audio\classical\961897.opus
Extracted: 778565 -> MTG_Jamendo_50\audio\orchestral\778565.opus

Final metadata validation passed.
No missing values.
No empty genres_list / instruments_list / moods_list.
All audio files exist.

Saved final metadata: MTG_Jamendo_50\mtg_jamendo_50_metadata.csv
Tracks: 60

Selected genre distribution:
selected_genre
soundtrack       6
electronic       6
ambient          6
pop              6
classical        6
easylistening    6
rock             6
orchestral       6
chillout       

## Загрузка видео файлов с разметкой

In [2]:
import re
import shutil
import subprocess
from pathlib import Path
from typing import Optional

import pandas as pd
from tqdm import tqdm


# =========================
# CONFIG
# =========================

OPENLAV_DIR = Path("OpenLAV")
OPENLAV_CSV = OPENLAV_DIR / "video_database.csv"

OPENLAV_RAW_VIDEO_DIR = OPENLAV_DIR / "raw_videos"
OPENLAV_TRIMMED_VIDEO_DIR = OPENLAV_DIR / "videos"

OPENLAV_REPORT = OPENLAV_DIR / "openlav_video_download_report.csv"


# =========================
# UTILS
# =========================

def read_csv_safely(path: Path) -> pd.DataFrame:
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
    last_error = None

    for encoding in encodings:
        try:
            print(f"Trying encoding: {encoding}")
            df = pd.read_csv(
                path,
                sep=None,
                engine="python",
                encoding=encoding,
            )
            print(f"Successfully read CSV with encoding: {encoding}")
            return df
        except UnicodeDecodeError as e:
            last_error = e

    raise last_error


def safe_filename(text: str, max_len: int = 80) -> str:
    text = str(text).strip()
    text = re.sub(r"[^\w\-а-яА-ЯёЁ]+", "_", text, flags=re.UNICODE)
    text = re.sub(r"_+", "_", text)
    text = text[:max_len].strip("_")
    return text or "untitled"


def normalize_title(title) -> str:
    title = str(title).strip().lower()
    title = re.sub(r"\s+", " ", title)
    return title


def normalize_url(url) -> str:
    url = str(url).strip().lower()
    return url


def parse_duration_seconds(value) -> Optional[float]:
    if pd.isna(value):
        return None

    value = str(value).strip().replace(",", ".")

    try:
        duration = float(value)
        if duration <= 0:
            return None
        return duration
    except ValueError:
        return None


def find_existing_file(out_dir: Path, base_name: str) -> Optional[Path]:
    matches = list(out_dir.glob(f"{base_name}.*"))
    if matches:
        return matches[0]
    return None


def has_ffmpeg() -> bool:
    return shutil.which("ffmpeg") is not None


def trim_video_ffmpeg(
    input_path: Path,
    output_path: Path,
    duration_s: float,
    overwrite: bool = True,
) -> bool:
    """
    Обрезает видео от начала до duration_s.

    Сначала пробуем stream copy (-c copy), это быстро.
    Если не получилось, пробуем перекодирование libx264/aac.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not overwrite:
        return True

    if output_path.exists() and overwrite:
        output_path.unlink()

    # Быстрая обрезка без перекодирования.
    cmd_copy = [
        "ffmpeg",
        "-y",
        "-i", str(input_path),
        "-t", str(duration_s),
        "-c", "copy",
        "-avoid_negative_ts", "make_zero",
        str(output_path),
    ]

    result = subprocess.run(
        cmd_copy,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    if result.returncode == 0 and output_path.exists() and output_path.stat().st_size > 0:
        return True

    # Если stream copy не сработал, перекодируем.
    if output_path.exists():
        output_path.unlink()

    cmd_reencode = [
        "ffmpeg",
        "-y",
        "-i", str(input_path),
        "-t", str(duration_s),
        "-vf", "scale='min(720,iw)':-2",
        "-c:v", "libx264",
        "-preset", "veryfast",
        "-crf", "23",
        "-c:a", "aac",
        "-b:a", "128k",
        "-movflags", "+faststart",
        str(output_path),
    ]

    result = subprocess.run(
        cmd_reencode,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    return result.returncode == 0 and output_path.exists() and output_path.stat().st_size > 0


# =========================
# MAIN DOWNLOADER
# =========================

def download_openlav_videos(
    csv_path: Path = OPENLAV_CSV,
    raw_dir: Path = OPENLAV_RAW_VIDEO_DIR,
    trimmed_dir: Path = OPENLAV_TRIMMED_VIDEO_DIR,
    report_path: Path = OPENLAV_REPORT,
    max_videos: Optional[int] = None,
    deduplicate: bool = True,
    trim_to_annotation_length: bool = True,
) -> pd.DataFrame:
    """
    Скачивает OpenLAV видео и обрезает их до length_s из разметки.

    На выходе:
        OpenLAV/raw_videos/  — исходные скачанные видео
        OpenLAV/videos/      — обрезанные видео
        OpenLAV/openlav_video_download_report.csv
    """
    try:
        import yt_dlp
    except ImportError:
        raise ImportError("Установи yt-dlp: pip install yt-dlp")

    if not csv_path.exists():
        raise FileNotFoundError(f"Не найден файл: {csv_path}")

    raw_dir.mkdir(parents=True, exist_ok=True)
    trimmed_dir.mkdir(parents=True, exist_ok=True)
    report_path.parent.mkdir(parents=True, exist_ok=True)

    if trim_to_annotation_length and not has_ffmpeg():
        raise RuntimeError(
            "Для обрезки видео нужен ffmpeg, но он не найден в PATH. "
            "Установи ffmpeg или запусти с trim_to_annotation_length=False."
        )

    df = read_csv_safely(csv_path)
    df.columns = [str(c).strip() for c in df.columns]

    required = {"video_number", "video_title", "source_URL", "length_s"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"В OpenLAV CSV не найдены столбцы: {missing}. "
            f"Есть столбцы: {list(df.columns)}"
        )

    before = len(df)

    df["video_title_norm"] = df["video_title"].apply(normalize_title)
    df["source_url_norm"] = df["source_URL"].apply(normalize_url)

    if deduplicate:
        df = df.drop_duplicates(subset=["source_url_norm"], keep="first")
        df = df.drop_duplicates(subset=["video_title_norm"], keep="first")
        df = df.copy()

    after = len(df)
    print(f"Deduplication: {before} -> {after}")

    if max_videos is not None:
        df = df.head(max_videos)

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="OpenLAV videos"):
        video_number = str(row.get("video_number", "")).strip()
        video_title = str(row.get("video_title", "")).strip()
        url = str(row.get("source_URL", "")).strip()
        duration_s = parse_duration_seconds(row.get("length_s", None))

        if not video_number or video_number.lower() == "nan":
            video_number = str(len(results) + 1)

        if not video_title or video_title.lower() == "nan":
            video_title = "untitled"

        if not url or url.lower() == "nan":
            results.append({
                "video_number": video_number,
                "video_title": video_title,
                "source_URL": url,
                "length_s": duration_s,
                "status": "skipped_no_url",
                "raw_file_path": "",
                "trimmed_file_path": "",
                "error": "",
            })
            continue

        base_name = f"{safe_filename(video_number, 20)}_{safe_filename(video_title, 80)}"

        raw_existing = find_existing_file(raw_dir, base_name)
        trimmed_path = trimmed_dir / f"{base_name}.mp4"

        # Если уже есть обрезанная версия, пропускаем.
        if trimmed_path.exists():
            results.append({
                "video_number": video_number,
                "video_title": video_title,
                "source_URL": url,
                "length_s": duration_s,
                "status": "already_exists_trimmed",
                "raw_file_path": str(raw_existing) if raw_existing else "",
                "trimmed_file_path": str(trimmed_path),
                "error": "",
            })
            continue

        # Если raw уже есть, не скачиваем заново.
        raw_file = raw_existing

        if raw_file is None:
            output_template = str(raw_dir / f"{base_name}.%(ext)s")

            ydl_opts = {
                "outtmpl": output_template,

                # Вариант без обязательного склеивания через ffmpeg.
                # Качество ниже, зато меньше проблем.
                "format": "best[ext=mp4]/best",

                "noplaylist": True,
                "quiet": False,
                "no_warnings": False,
                "ignoreerrors": False,
                "restrictfilenames": True,
                "retries": 3,
                "fragment_retries": 3,
            }

            try:
                with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                    ydl.extract_info(url, download=True)

                raw_file = find_existing_file(raw_dir, base_name)

                if raw_file is None:
                    results.append({
                        "video_number": video_number,
                        "video_title": video_title,
                        "source_URL": url,
                        "length_s": duration_s,
                        "status": "failed_download_no_file",
                        "raw_file_path": "",
                        "trimmed_file_path": "",
                        "error": "yt-dlp finished, but no raw file was found",
                    })
                    continue

            except Exception as e:
                results.append({
                    "video_number": video_number,
                    "video_title": video_title,
                    "source_URL": url,
                    "length_s": duration_s,
                    "status": "failed_download_exception",
                    "raw_file_path": "",
                    "trimmed_file_path": "",
                    "error": str(e),
                })
                continue

        # Обрезка.
        if trim_to_annotation_length:
            if duration_s is None:
                results.append({
                    "video_number": video_number,
                    "video_title": video_title,
                    "source_URL": url,
                    "length_s": duration_s,
                    "status": "downloaded_not_trimmed_no_duration",
                    "raw_file_path": str(raw_file),
                    "trimmed_file_path": "",
                    "error": "length_s is missing or invalid",
                })
                continue

            try:
                ok = trim_video_ffmpeg(
                    input_path=raw_file,
                    output_path=trimmed_path,
                    duration_s=duration_s,
                    overwrite=True,
                )

                if ok:
                    results.append({
                        "video_number": video_number,
                        "video_title": video_title,
                        "source_URL": url,
                        "length_s": duration_s,
                        "status": "downloaded_and_trimmed",
                        "raw_file_path": str(raw_file),
                        "trimmed_file_path": str(trimmed_path),
                        "error": "",
                    })
                else:
                    results.append({
                        "video_number": video_number,
                        "video_title": video_title,
                        "source_URL": url,
                        "length_s": duration_s,
                        "status": "downloaded_trim_failed",
                        "raw_file_path": str(raw_file),
                        "trimmed_file_path": "",
                        "error": "ffmpeg failed to trim video",
                    })

            except Exception as e:
                results.append({
                    "video_number": video_number,
                    "video_title": video_title,
                    "source_URL": url,
                    "length_s": duration_s,
                    "status": "downloaded_trim_exception",
                    "raw_file_path": str(raw_file),
                    "trimmed_file_path": "",
                    "error": str(e),
                })

        else:
            results.append({
                "video_number": video_number,
                "video_title": video_title,
                "source_URL": url,
                "length_s": duration_s,
                "status": "downloaded_raw_only",
                "raw_file_path": str(raw_file),
                "trimmed_file_path": "",
                "error": "",
            })

    report = pd.DataFrame(results)
    report.to_csv(report_path, index=False, encoding="utf-8-sig")

    print("\nSaved OpenLAV download report:", report_path)
    print("\nStatus counts:")
    print(report["status"].value_counts(dropna=False))

    return report


# =========================
# MAIN
# =========================

if __name__ == "__main__":
    print("=== Downloading and trimming OpenLAV videos ===")

    report = download_openlav_videos(
        csv_path=OPENLAV_CSV,
        raw_dir=OPENLAV_RAW_VIDEO_DIR,
        trimmed_dir=OPENLAV_TRIMMED_VIDEO_DIR,
        report_path=OPENLAV_REPORT,
        max_videos=None,
        deduplicate=True,
        trim_to_annotation_length=True,
    )

    print("\nDone.")

=== Downloading and trimming OpenLAV videos ===
Trying encoding: utf-8
Trying encoding: utf-8-sig
Trying encoding: cp1252
Successfully read CSV with encoding: cp1252
Deduplication: 188 -> 172


OpenLAV videos:   0%|          | 0/172 [00:00<?, ?it/s]

[youtube] Extracting URL: https://www.youtube.com/watch?v=FN60fhzA5YE
[youtube] FN60fhzA5YE: Downloading webpage


[youtube] FN60fhzA5YE: Downloading android vr player API JSON


ERROR: [youtube] FN60fhzA5YE: Video unavailable
OpenLAV videos:   3%|▎         | 5/172 [00:00<00:29,  5.71it/s]

...

ERROR: [youtube] Yie9Bgo69m8: This video is not available
OpenLAV videos:  94%|█████████▍| 162/172 [01:00<00:04,  2.08it/s]

[youtube] Extracting URL: https://www.youtube.com/watch?v=9TvIyIchWjg
[youtube] 9TvIyIchWjg: Downloading webpage


[youtube] 9TvIyIchWjg: Downloading android vr player API JSON


ERROR: [youtube] 9TvIyIchWjg: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
OpenLAV videos: 100%|██████████| 172/172 [01:01<00:00,  2.80it/s]


Saved OpenLAV download report: OpenLAV\openlav_video_download_report.csv

Status counts:
status
already_exists_trimmed       118
failed_download_exception     54
Name: count, dtype: int64

Done.


# Формирование датасета video - audio track

In [11]:
import ast
import json
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd
from sklearn.model_selection import train_test_split


# =========================
# PATHS
# =========================

OPENLAV_CSV = Path("OpenLAV/video_database.csv")
OPENLAV_DOWNLOAD_REPORT = Path("OpenLAV/openlav_video_download_report.csv")

MTG_META_CSV = Path("MTG_Jamendo_50/mtg_jamendo_50_metadata.csv")

OUT_DIR = Path("final_dataset")
OUT_CSV = OUT_DIR / "video_music_pairs.csv"
OUT_TRAIN_CSV = OUT_DIR / "train.csv"
OUT_VAL_CSV = OUT_DIR / "val.csv"
OUT_TEST_CSV = OUT_DIR / "test.csv"

MAPPING_CSV = OUT_DIR / "emotion_to_audio_category_mapping.csv"
AUDIO_KEYWORDS_CSV = OUT_DIR / "audio_category_keywords.csv"
TRACKS_WITH_CATEGORIES_CSV = OUT_DIR / "tracks_with_audio_categories.csv"
VIDEOS_CLEAN_CSV = OUT_DIR / "videos_clean.csv"


# =========================
# SETTINGS
# =========================

POSITIVE_THRESHOLD = 0.5

USE_ONLY_DOWNLOADED_VIDEOS = True

MAX_VIDEOS = None
MAX_TRACKS = None

RANDOM_STATE = 42
TRAIN_SIZE = 0.7
VAL_SIZE = 0.15
TEST_SIZE = 0.15


# =========================
# OPENLAV EMOTIONS
# =========================

KNOWN_OPENLAV_EMOTIONS = {
    "anger",
    "compassion",
    "disgust",
    "fascination",
    "fear",
    "joy",
    "no_emotion",
    "sadness",
    "satisfaction",
    "surprise",
}

# Если emotion_code в OpenLAV числовой, используется этот порядок.
# Если emotion_code уже строковый, например "joy", код сам это обработает.
EMOTION_CODE_TO_LABEL = {
    "0": "anger",
    "1": "compassion",
    "2": "disgust",
    "3": "fascination",
    "4": "fear",
    "5": "joy",
    "6": "no_emotion",
    "7": "sadness",
    "8": "satisfaction",
    "9": "surprise",
    0: "anger",
    1: "compassion",
    2: "disgust",
    3: "fascination",
    4: "fear",
    5: "joy",
    6: "no_emotion",
    7: "sadness",
    8: "satisfaction",
    9: "surprise",
}


# =========================
# AUDIO CATEGORIES
# =========================

AUDIO_CATEGORY_KEYWORDS: Dict[str, List[str]] = {
    # Весёлая, позитивная, яркая музыка.
    # Сюда лучше относить pop/funk/reggae/disco, но не всё dance/electronic.
    "happy_upbeat": [
        "happy",
        "positive",
        "upbeat",
        "uplifting",
        "fun",
        "summer",
        "holiday",
        "party",
        "pop",
        "funk",
        "disco",
        "reggae",
        "soul",
        "groovy",
        "cool",
    ],

    # Энергичная электронная музыка.
    # Здесь оставляем именно electronic/club/fast/action.
    "energetic_electronic": [
        "electronic",
        "techno",
        "house",
        "trance",
        "edm",
        "dance",
        "club",
        "beats",
        "synth",
        "energetic",
        "energy",
        "fast",
        "sport",
        "action",
        "motivational",
    ],

    # Спокойная фоновая/расслабляющая музыка.
    # Важно: ambient убран отсюда, чтобы он уходил в dreamy_atmospheric.
    "calm_relaxing": [
        "calm",
        "relaxing",
        "relax",
        "meditation",
        "meditative",
        "soft",
        "slow",
        "chill",
        "chillout",
        "acoustic",
        "folk",
        "folkcountry",
        "classical",
        "easy",
        "peaceful",
    ],

    # Грустная, меланхоличная, эмоциональная.
    "sad_melancholic": [
        "sad",
        "melancholic",
        "melancholy",
        "emotional",
        "ballad",
        "drama",
        "dramatic",
        "blues",
        "piano",
        "lonely",
        "nostalgic",
    ],

    # Тёмная, тревожная, напряжённая, кинематографичная.
    # Сюда добавлены soundtrack/film/movie/trailer, чтобы такие треки
    # не улетали в energetic_electronic или neutral_background.
    "dark_tense": [
        "dark",
        "tense",
        "tension",
        "fear",
        "horror",
        "suspense",
        "thriller",
        "scary",
        "dramatic",
        "cinematic",
        "soundtrack",
        "trailer",
        "film",
        "movie",
        "epic",
        "industrial",
        "mysterious",
    ],

    # Агрессивная/интенсивная музыка.
    # Здесь рок/метал/панк должны перехватываться уверенно.
    "aggressive_intense": [
        "aggressive",
        "intense",
        "powerful",
        "heavy",
        "hard",
        "rock",
        "metal",
        "punk",
        "hardrock",
        "hard_rock",
        "alternative",
        "grunge",
        "distortion",
    ],

    # Атмосферная, мечтательная, пространственная.
    # Ambient/soundscape/space/nature лучше держать здесь.
    "dreamy_atmospheric": [
        "dream",
        "dreamy",
        "atmospheric",
        "ambient",
        "soundscape",
        "space",
        "nature",
        "deep",
        "ethereal",
        "newage",
        "new_age",
        "adventure",
        "inspiring",
        "hope",
    ],

    # Нейтральная прикладная музыка.
    # Это не обязательно calm: скорее corporate/commercial/background.
    "neutral_background": [
        "background",
        "neutral",
        "corporate",
        "commercial",
        "advertising",
        "documentary",
        "minimal",
        "minimalistic",
        "instrumental",
        "presentation",
        "business",
        "tutorial",
    ],

    # Тёплая, романтическая, мягкая.
    "romantic_warm": [
        "romantic",
        "romance",
        "love",
        "warm",
        "gentle",
        "tender",
        "soft",
        "emotional",
        "acousticguitar",
        "acoustic_guitar",
        "guitar",
        "piano",
        "jazz",
        "smooth",
    ],

    # Игривая, комедийная, детская, забавная.
    "playful_fun": [
        "playful",
        "funny",
        "comedy",
        "children",
        "child",
        "kids",
        "game",
        "quirky",
        "cartoon",
        "circus",
        "toy",
        "light",
        "joke",
    ],
}

VIDEO_EMOTION_TO_AUDIO_SCORE: Dict[str, Dict[str, float]] = {
    "joy": {
        "happy_upbeat": 1.0,
        "playful_fun": 0.9,
        "energetic_electronic": 0.8,
        "romantic_warm": 0.6,
        "calm_relaxing": 0.4,
        "neutral_background": 0.4,
        "dreamy_atmospheric": 0.3,
        "sad_melancholic": 0.0,
        "dark_tense": 0.0,
        "aggressive_intense": 0.2,
    },

    "satisfaction": {
        "calm_relaxing": 1.0,
        "romantic_warm": 0.9,
        "neutral_background": 0.7,
        "happy_upbeat": 0.6,
        "dreamy_atmospheric": 0.6,
        "playful_fun": 0.3,
        "sad_melancholic": 0.2,
        "energetic_electronic": 0.2,
        "dark_tense": 0.0,
        "aggressive_intense": 0.0,
    },

    "fascination": {
        "dreamy_atmospheric": 1.0,
        "dark_tense": 0.7,
        "calm_relaxing": 0.6,
        "neutral_background": 0.5,
        "romantic_warm": 0.5,
        "energetic_electronic": 0.4,
        "happy_upbeat": 0.3,
        "sad_melancholic": 0.3,
        "playful_fun": 0.2,
        "aggressive_intense": 0.2,
    },

    "surprise": {
        "playful_fun": 1.0,
        "energetic_electronic": 0.8,
        "dark_tense": 0.7,
        "happy_upbeat": 0.6,
        "dreamy_atmospheric": 0.5,
        "aggressive_intense": 0.5,
        "neutral_background": 0.2,
        "calm_relaxing": 0.1,
        "romantic_warm": 0.1,
        "sad_melancholic": 0.1,
    },

    "sadness": {
        "sad_melancholic": 1.0,
        "calm_relaxing": 0.7,
        "dreamy_atmospheric": 0.7,
        "romantic_warm": 0.6,
        "neutral_background": 0.4,
        "dark_tense": 0.3,
        "aggressive_intense": 0.1,
        "happy_upbeat": 0.0,
        "playful_fun": 0.0,
        "energetic_electronic": 0.0,
    },

    "fear": {
        "dark_tense": 1.0,
        "dreamy_atmospheric": 0.7,
        "aggressive_intense": 0.6,
        "sad_melancholic": 0.4,
        "neutral_background": 0.3,
        "calm_relaxing": 0.1,
        "energetic_electronic": 0.1,
        "happy_upbeat": 0.0,
        "playful_fun": 0.0,
        "romantic_warm": 0.0,
    },

    "anger": {
        "aggressive_intense": 1.0,
        "dark_tense": 0.8,
        "energetic_electronic": 0.7,
        "sad_melancholic": 0.2,
        "neutral_background": 0.1,
        "dreamy_atmospheric": 0.1,
        "happy_upbeat": 0.0,
        "calm_relaxing": 0.0,
        "romantic_warm": 0.0,
        "playful_fun": 0.0,
    },

    "disgust": {
        "dark_tense": 1.0,
        "aggressive_intense": 0.8,
        "sad_melancholic": 0.4,
        "dreamy_atmospheric": 0.3,
        "neutral_background": 0.2,
        "energetic_electronic": 0.1,
        "happy_upbeat": 0.0,
        "calm_relaxing": 0.0,
        "romantic_warm": 0.0,
        "playful_fun": 0.0,
    },

    "compassion": {
        "romantic_warm": 1.0,
        "calm_relaxing": 0.8,
        "sad_melancholic": 0.8,
        "dreamy_atmospheric": 0.6,
        "neutral_background": 0.5,
        "happy_upbeat": 0.2,
        "playful_fun": 0.1,
        "energetic_electronic": 0.0,
        "dark_tense": 0.0,
        "aggressive_intense": 0.0,
    },

    "no_emotion": {
        "neutral_background": 1.0,
        "calm_relaxing": 0.8,
        "dreamy_atmospheric": 0.6,
        "happy_upbeat": 0.4,
        "romantic_warm": 0.4,
        "sad_melancholic": 0.3,
        "playful_fun": 0.3,
        "energetic_electronic": 0.2,
        "dark_tense": 0.1,
        "aggressive_intense": 0.0,
    },
}

# =========================
# UTILS
# =========================

def read_csv_safely(path: Path) -> pd.DataFrame:
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(path, sep=None, engine="python", encoding=encoding)
        except UnicodeDecodeError as e:
            last_error = e

    raise last_error


def normalize_text(value) -> str:
    if pd.isna(value):
        return ""

    value = str(value).strip().lower()
    value = value.replace("-", "_")
    value = re.sub(r"\s+", "_", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")


def normalize_title(value) -> str:
    if pd.isna(value):
        return ""

    value = str(value).strip().lower()
    value = re.sub(r"\s+", " ", value)
    return value


def parse_list_cell(value) -> List[str]:
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return [normalize_text(x) for x in value if str(x).strip()]

    value = str(value).strip()

    if not value:
        return []

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [normalize_text(x) for x in parsed if str(x).strip()]
    except Exception:
        pass

    return [normalize_text(x) for x in re.split(r"[,;|]", value) if x.strip()]


def parse_duration_seconds(value) -> Optional[float]:
    if pd.isna(value):
        return None

    value = str(value).strip().replace(",", ".")

    try:
        value = float(value)
    except ValueError:
        return None

    if value <= 0:
        return None

    return value


def get_openlav_emotion_label(value) -> str:
    if pd.isna(value):
        return "unknown"

    raw = value
    norm = normalize_text(value)

    if norm in KNOWN_OPENLAV_EMOTIONS:
        return norm

    if raw in EMOTION_CODE_TO_LABEL:
        return EMOTION_CODE_TO_LABEL[raw]

    if norm in EMOTION_CODE_TO_LABEL:
        return EMOTION_CODE_TO_LABEL[norm]

    try:
        as_int = int(float(str(value)))
        return EMOTION_CODE_TO_LABEL.get(as_int, "unknown")
    except Exception:
        return "unknown"


def get_audio_category_scores(row: pd.Series) -> Dict[str, int]:
    selected_genre = normalize_text(row.get("selected_genre", ""))

    genres = parse_list_cell(row.get("genres_list", []))
    moods = parse_list_cell(row.get("moods_list", []))
    instruments = parse_list_cell(row.get("instruments_list", []))

    all_tags = [selected_genre] + genres + moods + instruments
    all_tags = [tag for tag in all_tags if tag]

    scores = {}

    for category, keywords in AUDIO_CATEGORY_KEYWORDS.items():
        score = 0

        for tag in all_tags:
            for keyword in keywords:
                keyword = normalize_text(keyword)

                if tag == keyword:
                    score += 3
                elif keyword in tag or tag in keyword:
                    score += 1

        scores[category] = score

    return scores


def detect_audio_category(row: pd.Series) -> Tuple[str, Dict[str, int]]:
    scores = get_audio_category_scores(row)

    best_category = max(scores, key=scores.get)

    if scores[best_category] == 0:
        best_category = "neutral_background"

    return best_category, scores


def get_compatibility_score(video_emotion: str, audio_category: str) -> float:
    video_emotion = normalize_text(video_emotion)
    audio_category = normalize_text(audio_category)

    return float(
        VIDEO_EMOTION_TO_AUDIO_SCORE
        .get(video_emotion, {})
        .get(audio_category, 0.0)
    )


# =========================
# LOAD VIDEOS
# =========================

def load_openlav_videos() -> pd.DataFrame:
    if not OPENLAV_CSV.exists():
        raise FileNotFoundError(f"Не найден файл: {OPENLAV_CSV}")

    df = read_csv_safely(OPENLAV_CSV)
    df.columns = [str(c).strip() for c in df.columns]

    required = {"emotion_code", "video_number", "video_title", "length_s"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"В OpenLAV CSV не найдены столбцы: {missing}. "
            f"Есть столбцы: {list(df.columns)}"
        )

    df = df.copy()

    df["video_id"] = df["video_number"].astype(str).str.strip()
    df["video_title"] = df["video_title"].astype(str).str.strip()
    df["video_title_norm"] = df["video_title"].apply(normalize_title)

    df["a_priori_emotion"] = df["emotion_code"].apply(get_openlav_emotion_label)
    df["duration_s"] = df["length_s"].apply(parse_duration_seconds)

    # Дедупликация видео. В OpenLAV могут встречаться разные ссылки на один и тот же ролик.
    before = len(df)
    df = df.drop_duplicates(subset=["video_title_norm"], keep="first").copy()
    after = len(df)
    print(f"Video deduplication by title: {before} -> {after}")

    # Подмешиваем путь к реально скачанному/обрезанному видео.
    if USE_ONLY_DOWNLOADED_VIDEOS:
        if not OPENLAV_DOWNLOAD_REPORT.exists():
            raise FileNotFoundError(
                f"USE_ONLY_DOWNLOADED_VIDEOS=True, но не найден отчёт: {OPENLAV_DOWNLOAD_REPORT}"
            )

        report = pd.read_csv(OPENLAV_DOWNLOAD_REPORT)
        report.columns = [str(c).strip() for c in report.columns]

        if "video_number" not in report.columns:
            raise ValueError("В отчёте скачивания нет столбца video_number.")

        report["video_id"] = report["video_number"].astype(str).str.strip()

        # Берём только успешно обрезанные или уже существующие обрезанные видео.
        good_statuses = {
            "downloaded_and_trimmed",
            "already_exists_trimmed",
        }

        if "status" not in report.columns:
            raise ValueError("В отчёте скачивания нет столбца status.")

        report = report[report["status"].isin(good_statuses)].copy()

        if "trimmed_file_path" in report.columns:
            report["video_path"] = report["trimmed_file_path"]
        elif "file_path" in report.columns:
            report["video_path"] = report["file_path"]
        else:
            raise ValueError(
                "В отчёте скачивания нет ни trimmed_file_path, ни file_path."
            )

        report = report[["video_id", "video_path", "status"]].drop_duplicates(
            subset=["video_id"],
            keep="first",
        )

        df = df.merge(report, on="video_id", how="inner")

    else:
        df["video_path"] = ""
        df["status"] = "not_checked"

    df = df[df["a_priori_emotion"] != "unknown"].copy()
    df = df[df["duration_s"].notna()].copy()

    keep_cols = [
        "video_id",
        "video_title",
        "duration_s",
        "a_priori_emotion",
        "video_path",
    ]

    df = df[keep_cols].copy()

    if MAX_VIDEOS is not None:
        df = df.head(MAX_VIDEOS).copy()

    if df.empty:
        raise ValueError("После фильтрации не осталось видео.")

    return df


# =========================
# LOAD TRACKS
# =========================

def load_tracks() -> pd.DataFrame:
    if not MTG_META_CSV.exists():
        raise FileNotFoundError(f"Не найден файл: {MTG_META_CSV}")

    df = pd.read_csv(MTG_META_CSV)
    df.columns = [str(c).strip() for c in df.columns]

    required = {"id", "selected_genre"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"В MTG metadata CSV не найдены столбцы: {missing}. "
            f"Есть столбцы: {list(df.columns)}"
        )

    df = df.copy()
    df["track_id"] = df["id"].astype(str).str.strip()

    categories = []
    debug_scores = []

    for _, row in df.iterrows():
        category, scores = detect_audio_category(row)
        categories.append(category)
        debug_scores.append(json.dumps(scores, ensure_ascii=False))

    df["audio_category"] = categories
    df["audio_category_debug_scores"] = debug_scores

    if "audio_path" not in df.columns:
        df["audio_path"] = ""

    keep_cols = [
        "track_id",
        "selected_genre",
        "audio_category",
        "audio_category_debug_scores",
        "audio_path",
    ]

    for col in ["genres_list", "moods_list", "instruments_list"]:
        if col in df.columns:
            keep_cols.append(col)

    df = df[keep_cols].drop_duplicates(subset=["track_id"], keep="first").copy()

    if MAX_TRACKS is not None:
        df = df.head(MAX_TRACKS).copy()

    if df.empty:
        raise ValueError("После фильтрации не осталось треков.")

    return df


# =========================
# BUILD PAIRS
# =========================

def build_pairs(videos: pd.DataFrame, tracks: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, video in videos.iterrows():
        for _, track in tracks.iterrows():
            score = get_compatibility_score(
                video_emotion=video["a_priori_emotion"],
                audio_category=track["audio_category"],
            )

            rows.append({
                "video_id": video["video_id"],
                "video_title": video["video_title"],
                "video_path": video["video_path"],
                "duration_s": video["duration_s"],
                "a_priori_emotion": video["a_priori_emotion"],

                "track_id": track["track_id"],
                "audio_path": track.get("audio_path", ""),
                "selected_genre": track.get("selected_genre", ""),
                "audio_category": track["audio_category"],

                "compatibility_score": score,
                "target_binary": int(score >= POSITIVE_THRESHOLD),
            })

    pairs = pd.DataFrame(rows)

    if pairs.empty:
        raise ValueError("Итоговый датасет пар пустой.")

    return pairs


# =========================
# SPLIT
# =========================

def split_by_video_id(pairs: pd.DataFrame) -> pd.DataFrame:
    """
    Методологически важно делить не строки, а video_id.
    Иначе одно и то же видео попадёт и в train, и в test с разными треками.
    """
    video_ids = pairs["video_id"].drop_duplicates().tolist()

    train_ids, temp_ids = train_test_split(
        video_ids,
        train_size=TRAIN_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True,
    )

    relative_val_size = VAL_SIZE / (VAL_SIZE + TEST_SIZE)

    val_ids, test_ids = train_test_split(
        temp_ids,
        train_size=relative_val_size,
        random_state=RANDOM_STATE,
        shuffle=True,
    )

    split_map = {}

    for video_id in train_ids:
        split_map[video_id] = "train"

    for video_id in val_ids:
        split_map[video_id] = "val"

    for video_id in test_ids:
        split_map[video_id] = "test"

    pairs = pairs.copy()
    pairs["split"] = pairs["video_id"].map(split_map)

    return pairs


# =========================
# SAVE AUXILIARY TABLES
# =========================

def save_mapping_tables(tracks: pd.DataFrame, videos: pd.DataFrame) -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    mapping_rows = []

    for emotion, category_scores in VIDEO_EMOTION_TO_AUDIO_SCORE.items():
        for category, score in category_scores.items():
            mapping_rows.append({
                "a_priori_emotion": emotion,
                "audio_category": category,
                "compatibility_score": score,
                "target_binary": int(score >= POSITIVE_THRESHOLD),
            })

    pd.DataFrame(mapping_rows).to_csv(MAPPING_CSV, index=False, encoding="utf-8-sig")

    keyword_rows = []

    for category, keywords in AUDIO_CATEGORY_KEYWORDS.items():
        for keyword in keywords:
            keyword_rows.append({
                "audio_category": category,
                "keyword": keyword,
            })

    pd.DataFrame(keyword_rows).to_csv(AUDIO_KEYWORDS_CSV, index=False, encoding="utf-8-sig")

    tracks.to_csv(TRACKS_WITH_CATEGORIES_CSV, index=False, encoding="utf-8-sig")
    videos.to_csv(VIDEOS_CLEAN_CSV, index=False, encoding="utf-8-sig")


# =========================
# STATS
# =========================

def print_stats(videos: pd.DataFrame, tracks: pd.DataFrame, pairs: pd.DataFrame) -> None:
    print("\n=== Videos ===")
    print(f"Videos: {len(videos)}")
    print(videos["a_priori_emotion"].value_counts(dropna=False))

    print("\n=== Tracks ===")
    print(f"Tracks: {len(tracks)}")
    print(tracks["audio_category"].value_counts(dropna=False))

    print("\n=== Pairs ===")
    print(f"Pairs: {len(pairs)}")
    print(f"Positive pairs: {pairs['target_binary'].sum()}")
    print(f"Negative pairs: {(pairs['target_binary'] == 0).sum()}")
    print(f"Positive ratio: {pairs['target_binary'].mean():.3f}")

    print("\n=== Split sizes ===")
    print(pairs.groupby("split")["video_id"].nunique().rename("unique_videos"))
    print(pairs["split"].value_counts())

    print("\n=== Positive ratio by split ===")
    print(pairs.groupby("split")["target_binary"].mean())

    print("\n=== Positive ratio by emotion ===")
    print(
        pairs.groupby("a_priori_emotion")["target_binary"]
        .agg(["count", "sum", "mean"])
        .rename(columns={"sum": "positive_count", "mean": "positive_ratio"})
        .sort_values("count", ascending=False)
    )

    print("\n=== Positive ratio by audio_category ===")
    print(
        pairs.groupby("audio_category")["target_binary"]
        .agg(["count", "sum", "mean"])
        .rename(columns={"sum": "positive_count", "mean": "positive_ratio"})
        .sort_values("count", ascending=False)
    )


# =========================
# MAIN
# =========================

def main() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    videos = load_openlav_videos()
    tracks = load_tracks()

    pairs = build_pairs(videos, tracks)
    pairs = split_by_video_id(pairs)

    pairs.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

    pairs[pairs["split"] == "train"].to_csv(OUT_TRAIN_CSV, index=False, encoding="utf-8-sig")
    pairs[pairs["split"] == "val"].to_csv(OUT_VAL_CSV, index=False, encoding="utf-8-sig")
    pairs[pairs["split"] == "test"].to_csv(OUT_TEST_CSV, index=False, encoding="utf-8-sig")

    save_mapping_tables(tracks=tracks, videos=videos)
    print_stats(videos=videos, tracks=tracks, pairs=pairs)

    print("\nSaved:")
    print(f"  {OUT_CSV}")
    print(f"  {OUT_TRAIN_CSV}")
    print(f"  {OUT_VAL_CSV}")
    print(f"  {OUT_TEST_CSV}")
    print(f"  {MAPPING_CSV}")
    print(f"  {AUDIO_KEYWORDS_CSV}")
    print(f"  {TRACKS_WITH_CATEGORIES_CSV}")
    print(f"  {VIDEOS_CLEAN_CSV}")


if __name__ == "__main__":
    main()

Video deduplication by title: 188 -> 173

=== Videos ===
Videos: 98
a_priori_emotion
no_emotion      16
sadness         14
joy             14
disgust         13
satisfaction    13
compassion      12
surprise         8
fascination      4
fear             4
Name: count, dtype: int64

=== Tracks ===
Tracks: 60
audio_category
calm_relaxing           20
dreamy_atmospheric      12
dark_tense               8
aggressive_intense       7
happy_upbeat             6
sad_melancholic          3
energetic_electronic     3
romantic_warm            1
Name: count, dtype: int64

=== Pairs ===
Pairs: 5880
Positive pairs: 2850
Negative pairs: 3030
Positive ratio: 0.485

=== Split sizes ===
split
test     15
train    68
val      15
Name: unique_videos, dtype: int64
split
train    4080
test      900
val       900
Name: count, dtype: int64

=== Positive ratio by split ===
split
test     0.427778
train    0.477451
val      0.574444
Name: target_binary, dtype: float64

=== Positive ratio by emotion ===
        

# EDA of dataset video --> audio track

In [12]:
import ast
import json
import re
from pathlib import Path
from typing import List

import matplotlib.pyplot as plt
import pandas as pd


# =========================
# PATHS
# =========================

DATASET_CSV = Path("final_dataset/video_music_pairs.csv")
VIDEOS_CLEAN_CSV = Path("final_dataset/videos_clean.csv")
TRACKS_CSV = Path("final_dataset/tracks_with_audio_categories.csv")
MAPPING_CSV = Path("final_dataset/emotion_to_audio_category_mapping.csv")

OUT_DIR = Path("eda_report")
FIG_DIR = OUT_DIR / "figures"
TABLE_DIR = OUT_DIR / "tables"


# =========================
# CONFIG
# =========================

MIN_AUDIO_CATEGORY_COUNT = 3
MIN_VIDEO_EMOTION_COUNT = 3

EXPECTED_COLUMNS = [
    "video_id",
    "video_title",
    "video_path",
    "duration_s",
    "a_priori_emotion",
    "track_id",
    "audio_path",
    "selected_genre",
    "audio_category",
    "compatibility_score",
    "target_binary",
    "split",
]


# =========================
# UTILS
# =========================

def ensure_dirs() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    TABLE_DIR.mkdir(parents=True, exist_ok=True)


def read_csv_safely(path: Path) -> pd.DataFrame:
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError as e:
            last_error = e

    raise last_error


def file_exists(path_value) -> bool:
    if pd.isna(path_value):
        return False

    path_value = str(path_value).strip()

    if not path_value:
        return False

    path = Path(path_value)

    return path.exists() and path.is_file() and path.stat().st_size > 0


def save_bar(
    series: pd.Series,
    title: str,
    xlabel: str,
    ylabel: str,
    out_path: Path,
    rotation: int = 45,
) -> None:
    plt.figure(figsize=(10, 5))
    series.plot(kind="bar")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=rotation, ha="right")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


def save_hist(
    values: pd.Series,
    title: str,
    xlabel: str,
    ylabel: str,
    out_path: Path,
    bins: int = 20,
) -> None:
    values = pd.to_numeric(values, errors="coerce").dropna()

    plt.figure(figsize=(8, 5))
    plt.hist(values, bins=bins)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


def save_heatmap_table(
    table: pd.DataFrame,
    title: str,
    out_path: Path,
) -> None:
    plt.figure(figsize=(12, 6))
    plt.imshow(table.values, aspect="auto")
    plt.colorbar(label="value")
    plt.title(title)
    plt.xticks(range(len(table.columns)), table.columns, rotation=45, ha="right")
    plt.yticks(range(len(table.index)), table.index)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


def parse_list_cell(value) -> List[str]:
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    value = str(value).strip()

    if not value or value.lower() in {"nan", "none", "null", "[]"}:
        return []

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass

    try:
        parsed = json.loads(value)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass

    return [x.strip() for x in re.split(r"[,;|]", value) if x.strip()]


# =========================
# LOAD DATA
# =========================

def load_data():
    if not DATASET_CSV.exists():
        raise FileNotFoundError(f"Не найден основной датасет: {DATASET_CSV}")

    pairs = read_csv_safely(DATASET_CSV)
    pairs.columns = [str(c).strip() for c in pairs.columns]

    missing = set(EXPECTED_COLUMNS) - set(pairs.columns)
    if missing:
        raise ValueError(f"В основном датасете не хватает столбцов: {missing}")

    videos = None
    tracks = None
    mapping = None

    if VIDEOS_CLEAN_CSV.exists():
        videos = read_csv_safely(VIDEOS_CLEAN_CSV)
        videos.columns = [str(c).strip() for c in videos.columns]
    else:
        videos = pairs[
            ["video_id", "video_title", "video_path", "duration_s", "a_priori_emotion"]
        ].drop_duplicates(subset=["video_id"])

    if TRACKS_CSV.exists():
        tracks = read_csv_safely(TRACKS_CSV)
        tracks.columns = [str(c).strip() for c in tracks.columns]
    else:
        tracks = pairs[
            ["track_id", "audio_path", "selected_genre", "audio_category"]
        ].drop_duplicates(subset=["track_id"])

    if MAPPING_CSV.exists():
        mapping = read_csv_safely(MAPPING_CSV)
        mapping.columns = [str(c).strip() for c in mapping.columns]

    pairs["target_binary"] = pd.to_numeric(pairs["target_binary"], errors="coerce").fillna(0).astype(int)
    pairs["compatibility_score"] = pd.to_numeric(pairs["compatibility_score"], errors="coerce")
    pairs["duration_s"] = pd.to_numeric(pairs["duration_s"], errors="coerce")

    videos["duration_s"] = pd.to_numeric(videos["duration_s"], errors="coerce")

    return pairs, videos, tracks, mapping


# =========================
# BASIC CHECKS
# =========================

def run_basic_checks(pairs: pd.DataFrame, videos: pd.DataFrame, tracks: pd.DataFrame) -> pd.DataFrame:
    checks = []

    def add_check(name: str, value, status: str, comment: str = ""):
        checks.append({
            "check": name,
            "value": value,
            "status": status,
            "comment": comment,
        })

    add_check("num_pairs", len(pairs), "info")
    add_check("num_unique_videos", pairs["video_id"].nunique(), "info")
    add_check("num_unique_tracks", pairs["track_id"].nunique(), "info")

    duplicate_pairs = pairs.duplicated(subset=["video_id", "track_id"]).sum()
    add_check(
        "duplicate_video_track_pairs",
        duplicate_pairs,
        "ok" if duplicate_pairs == 0 else "warning",
        "Дубликаты пар video_id-track_id нежелательны.",
    )

    missing_values = pairs[EXPECTED_COLUMNS].isna().sum()
    total_missing = int(missing_values.sum())
    add_check(
        "total_missing_values_in_expected_columns",
        total_missing,
        "ok" if total_missing == 0 else "warning",
        "Если есть пропуски, проверь таблицу missing_values.csv.",
    )

    positive_ratio = pairs["target_binary"].mean()
    add_check(
        "positive_ratio",
        round(float(positive_ratio), 4),
        "ok" if 0.25 <= positive_ratio <= 0.6 else "warning",
        "Для retrieval/ranking обычно нормально, если positives не слишком редкие.",
    )

    video_counts = videos["a_priori_emotion"].value_counts()
    rare_emotions = video_counts[video_counts < MIN_VIDEO_EMOTION_COUNT].index.tolist()
    add_check(
        "rare_video_emotions",
        ", ".join(rare_emotions) if rare_emotions else "none",
        "ok" if not rare_emotions else "warning",
        f"Редкие эмоции: меньше {MIN_VIDEO_EMOTION_COUNT} видео.",
    )

    audio_counts = tracks["audio_category"].value_counts()
    rare_audio_categories = audio_counts[audio_counts < MIN_AUDIO_CATEGORY_COUNT].index.tolist()
    add_check(
        "rare_audio_categories",
        ", ".join(rare_audio_categories) if rare_audio_categories else "none",
        "ok" if not rare_audio_categories else "warning",
        f"Редкие аудиокатегории: меньше {MIN_AUDIO_CATEGORY_COUNT} треков.",
    )

    if "video_path" in videos.columns:
        video_exists = videos["video_path"].apply(file_exists)
        missing_video_files = int((~video_exists).sum())
        add_check(
            "missing_video_files",
            missing_video_files,
            "ok" if missing_video_files == 0 else "warning",
            "Часть video_path не существует или файл пустой.",
        )

    if "audio_path" in tracks.columns:
        audio_exists = tracks["audio_path"].apply(file_exists)
        missing_audio_files = int((~audio_exists).sum())
        add_check(
            "missing_audio_files",
            missing_audio_files,
            "ok" if missing_audio_files == 0 else "warning",
            "Часть audio_path не существует или файл пустой.",
        )

    if "split" in pairs.columns:
        leakage = (
            pairs.groupby("video_id")["split"]
            .nunique()
            .gt(1)
            .sum()
        )
        add_check(
            "video_id_split_leakage",
            int(leakage),
            "ok" if leakage == 0 else "error",
            "Одно видео не должно попадать в несколько split.",
        )

    checks_df = pd.DataFrame(checks)
    checks_df.to_csv(TABLE_DIR / "quality_checks.csv", index=False, encoding="utf-8-sig")

    missing_values.to_csv(TABLE_DIR / "missing_values.csv", encoding="utf-8-sig")

    return checks_df


# =========================
# TABLES
# =========================

def save_analysis_tables(pairs: pd.DataFrame, videos: pd.DataFrame, tracks: pd.DataFrame) -> None:
    videos["a_priori_emotion"].value_counts().to_csv(
        TABLE_DIR / "video_emotion_distribution.csv",
        encoding="utf-8-sig",
    )

    tracks["audio_category"].value_counts().to_csv(
        TABLE_DIR / "audio_category_distribution.csv",
        encoding="utf-8-sig",
    )

    pairs["target_binary"].value_counts().sort_index().to_csv(
        TABLE_DIR / "target_distribution.csv",
        encoding="utf-8-sig",
    )

    pairs.groupby("split")["target_binary"].agg(["count", "sum", "mean"]).rename(
        columns={"sum": "positive_count", "mean": "positive_ratio"}
    ).to_csv(TABLE_DIR / "positive_ratio_by_split.csv", encoding="utf-8-sig")

    pairs.groupby("a_priori_emotion")["target_binary"].agg(["count", "sum", "mean"]).rename(
        columns={"sum": "positive_count", "mean": "positive_ratio"}
    ).sort_values("count", ascending=False).to_csv(
        TABLE_DIR / "positive_ratio_by_emotion.csv",
        encoding="utf-8-sig",
    )

    pairs.groupby("audio_category")["target_binary"].agg(["count", "sum", "mean"]).rename(
        columns={"sum": "positive_count", "mean": "positive_ratio"}
    ).sort_values("count", ascending=False).to_csv(
        TABLE_DIR / "positive_ratio_by_audio_category.csv",
        encoding="utf-8-sig",
    )

    emotion_audio_score = pairs.pivot_table(
        index="a_priori_emotion",
        columns="audio_category",
        values="compatibility_score",
        aggfunc="mean",
    )

    emotion_audio_score.to_csv(
        TABLE_DIR / "mean_compatibility_score_emotion_x_audio_category.csv",
        encoding="utf-8-sig",
    )

    emotion_audio_target = pairs.pivot_table(
        index="a_priori_emotion",
        columns="audio_category",
        values="target_binary",
        aggfunc="mean",
    )

    emotion_audio_target.to_csv(
        TABLE_DIR / "positive_ratio_emotion_x_audio_category.csv",
        encoding="utf-8-sig",
    )

    if "genres_list" in tracks.columns:
        rows = []
        for _, row in tracks.iterrows():
            for tag in parse_list_cell(row["genres_list"]):
                rows.append({"track_id": row["track_id"], "tag": tag})

        if rows:
            pd.DataFrame(rows)["tag"].value_counts().to_csv(
                TABLE_DIR / "mtg_genre_tag_distribution.csv",
                encoding="utf-8-sig",
            )

    if "moods_list" in tracks.columns:
        rows = []
        for _, row in tracks.iterrows():
            for tag in parse_list_cell(row["moods_list"]):
                rows.append({"track_id": row["track_id"], "tag": tag})

        if rows:
            pd.DataFrame(rows)["tag"].value_counts().to_csv(
                TABLE_DIR / "mtg_mood_tag_distribution.csv",
                encoding="utf-8-sig",
            )

    if "instruments_list" in tracks.columns:
        rows = []
        for _, row in tracks.iterrows():
            for tag in parse_list_cell(row["instruments_list"]):
                rows.append({"track_id": row["track_id"], "tag": tag})

        if rows:
            pd.DataFrame(rows)["tag"].value_counts().to_csv(
                TABLE_DIR / "mtg_instrument_tag_distribution.csv",
                encoding="utf-8-sig",
            )


# =========================
# FIGURES
# =========================

def save_figures(pairs: pd.DataFrame, videos: pd.DataFrame, tracks: pd.DataFrame) -> None:
    save_bar(
        videos["a_priori_emotion"].value_counts(),
        title="Distribution of video emotions",
        xlabel="Video emotion",
        ylabel="Number of videos",
        out_path=FIG_DIR / "video_emotion_distribution.png",
    )

    save_bar(
        tracks["audio_category"].value_counts(),
        title="Distribution of audio categories",
        xlabel="Audio category",
        ylabel="Number of tracks",
        out_path=FIG_DIR / "audio_category_distribution.png",
    )

    save_bar(
        pairs["target_binary"].value_counts().sort_index(),
        title="Target distribution",
        xlabel="Target binary",
        ylabel="Number of pairs",
        out_path=FIG_DIR / "target_distribution.png",
        rotation=0,
    )

    save_hist(
        videos["duration_s"],
        title="Video duration distribution",
        xlabel="Duration, seconds",
        ylabel="Number of videos",
        out_path=FIG_DIR / "video_duration_distribution.png",
        bins=20,
    )

    split_positive = pairs.groupby("split")["target_binary"].mean().sort_index()
    save_bar(
        split_positive,
        title="Positive ratio by split",
        xlabel="Split",
        ylabel="Positive ratio",
        out_path=FIG_DIR / "positive_ratio_by_split.png",
        rotation=0,
    )

    emotion_positive = (
        pairs.groupby("a_priori_emotion")["target_binary"]
        .mean()
        .sort_values(ascending=False)
    )
    save_bar(
        emotion_positive,
        title="Positive ratio by video emotion",
        xlabel="Video emotion",
        ylabel="Positive ratio",
        out_path=FIG_DIR / "positive_ratio_by_emotion.png",
    )

    audio_positive = (
        pairs.groupby("audio_category")["target_binary"]
        .mean()
        .sort_values(ascending=False)
    )
    save_bar(
        audio_positive,
        title="Positive ratio by audio category",
        xlabel="Audio category",
        ylabel="Positive ratio",
        out_path=FIG_DIR / "positive_ratio_by_audio_category.png",
    )

    score_matrix = pairs.pivot_table(
        index="a_priori_emotion",
        columns="audio_category",
        values="compatibility_score",
        aggfunc="mean",
    ).fillna(0)

    save_heatmap_table(
        score_matrix,
        title="Mean compatibility score: emotion x audio category",
        out_path=FIG_DIR / "heatmap_mean_compatibility_score.png",
    )

    target_matrix = pairs.pivot_table(
        index="a_priori_emotion",
        columns="audio_category",
        values="target_binary",
        aggfunc="mean",
    ).fillna(0)

    save_heatmap_table(
        target_matrix,
        title="Positive ratio: emotion x audio category",
        out_path=FIG_DIR / "heatmap_positive_ratio.png",
    )

    if "durationInSec" in tracks.columns:
        save_hist(
            tracks["durationInSec"],
            title="Audio duration distribution",
            xlabel="Duration, seconds",
            ylabel="Number of tracks",
            out_path=FIG_DIR / "audio_duration_distribution.png",
            bins=20,
        )


# =========================
# TEXT SUMMARY
# =========================

def save_text_summary(pairs: pd.DataFrame, videos: pd.DataFrame, tracks: pd.DataFrame, checks: pd.DataFrame) -> None:
    summary_path = OUT_DIR / "eda_summary.txt"

    lines = []

    lines.append("EDA SUMMARY")
    lines.append("=" * 80)
    lines.append("")

    lines.append("Dataset size:")
    lines.append(f"- pairs: {len(pairs)}")
    lines.append(f"- unique videos: {pairs['video_id'].nunique()}")
    lines.append(f"- unique tracks: {pairs['track_id'].nunique()}")
    lines.append(f"- positive ratio: {pairs['target_binary'].mean():.3f}")
    lines.append("")

    lines.append("Video emotion distribution:")
    lines.append(str(videos["a_priori_emotion"].value_counts()))
    lines.append("")

    lines.append("Audio category distribution:")
    lines.append(str(tracks["audio_category"].value_counts()))
    lines.append("")

    lines.append("Positive ratio by split:")
    lines.append(str(pairs.groupby("split")["target_binary"].mean()))
    lines.append("")

    lines.append("Potential issues:")
    warnings = checks[checks["status"].isin(["warning", "error"])]

    if warnings.empty:
        lines.append("- No major issues detected.")
    else:
        for _, row in warnings.iterrows():
            lines.append(f"- [{row['status']}] {row['check']}: {row['value']} — {row['comment']}")

    lines.append("")
    lines.append("Interpretation:")
    lines.append(
        "- The dataset was formed as a weakly supervised video-music retrieval dataset. "
        "The main target is target_binary, derived from compatibility_score."
    )
    lines.append(
        "- The most important checks are class balance, audio category coverage, "
        "emotion coverage, missing files, and split leakage."
    )
    lines.append(
        "- If some audio categories or emotions are rare, they should be merged, removed, "
        "or supported by additional tracks/videos."
    )

    summary_path.write_text("\n".join(lines), encoding="utf-8")


# =========================
# MAIN
# =========================

def main() -> None:
    ensure_dirs()

    pairs, videos, tracks, mapping = load_data()

    checks = run_basic_checks(pairs, videos, tracks)
    save_analysis_tables(pairs, videos, tracks)
    save_figures(pairs, videos, tracks)
    save_text_summary(pairs, videos, tracks, checks)

    print("\nEDA finished.")
    print(f"Report directory: {OUT_DIR}")
    print(f"Figures: {FIG_DIR}")
    print(f"Tables: {TABLE_DIR}")

    print("\nQuality checks:")
    print(checks)

    print("\nMain dataset stats:")
    print(f"Pairs: {len(pairs)}")
    print(f"Videos: {pairs['video_id'].nunique()}")
    print(f"Tracks: {pairs['track_id'].nunique()}")
    print(f"Positive ratio: {pairs['target_binary'].mean():.3f}")


if __name__ == "__main__":
    main()


EDA finished.
Report directory: eda_report
Figures: eda_report\figures
Tables: eda_report\tables

Quality checks:
                                       check          value   status  \
0                                  num_pairs           5880     info   
1                          num_unique_videos             98     info   
2                          num_unique_tracks             60     info   
3                duplicate_video_track_pairs              0       ok   
4   total_missing_values_in_expected_columns              0       ok   
5                             positive_ratio         0.4847       ok   
6                        rare_video_emotions           none       ok   
7                      rare_audio_categories  romantic_warm  warning   
8                        missing_video_files              0       ok   
9                        missing_audio_files              0       ok   
10                    video_id_split_leakage              0       ok   

                    

# Формирование датасета user text query + video -> audio track

In [13]:
import random
import re
from pathlib import Path
from typing import Dict, List

import pandas as pd


# =========================
# PATHS
# =========================

IN_DIR = Path("final_dataset")
OUT_DIR = Path("final_dataset_with_queries")

TRAIN_IN = IN_DIR / "train.csv"
VAL_IN = IN_DIR / "val.csv"
TEST_IN = IN_DIR / "test.csv"

TRAIN_OUT = OUT_DIR / "train_with_queries.csv"
VAL_OUT = OUT_DIR / "val_with_queries.csv"
TEST_OUT = OUT_DIR / "test_with_queries.csv"


# =========================
# SETTINGS
# =========================

RANDOM_STATE = 42

# Вес видео и текстового запроса в итоговой weak-supervision разметке.
VIDEO_SCORE_WEIGHT = 0.6
QUERY_SCORE_WEIGHT = 0.4

# Порог для новой бинарной метки уже с учётом user_query.
POSITIVE_THRESHOLD = 0.6

# Сколько текстовых шаблонов брать на одну аудиокатегорию для одного видео.
# Если поставить больше, датасет быстро разрастётся.
TRAIN_TEMPLATES_PER_CATEGORY = 3
VAL_TEMPLATES_PER_CATEGORY = 2
TEST_TEMPLATES_PER_CATEGORY = 2

# Если True, то для каждого видео будут использоваться все аудиокатегории,
# которые реально есть в данном split.
# Это методологически лучше: для одного и того же видео пользователь может
# просить разные типы музыки.
USE_ALL_EXISTING_AUDIO_CATEGORIES_AS_QUERIES = True


# =========================
# HUGE QUERY TEMPLATES
# =========================

QUERY_TEMPLATES: Dict[str, List[str]] = {
    "happy_upbeat": [
        "хочу веселую позитивную музыку",
        "подбери радостный и бодрый трек",
        "нужна upbeat музыка для видео",
        "хочу что-то светлое и жизнерадостное",
        "подбери музыку с позитивным настроением",
        "нужен веселый трек без мрачной атмосферы",
        "хочу легкую радостную музыку",
        "подбери позитивную музыку для приятного видео",
        "нужен трек с хорошим настроением",
        "хочу музыку, которая звучит радостно и тепло",
        "подбери яркий и дружелюбный трек",
        "нужна музыка для счастливого момента",
        "хочу бодрую позитивную композицию",
        "подбери музыку для веселого ролика",
        "нужен uplifting трек",
        "хочу солнечную и приятную музыку",
        "подбери что-то радостное и ненавязчивое",
        "нужна позитивная музыка для короткого видео",
        "хочу трек с легкой праздничной атмосферой",
        "подбери музыку, которая создает хорошее настроение",
        "нужна cheerful музыка",
        "хочу приятный бодрый поп-трек",
        "подбери музыку для радостного настроения",
        "нужен оптимистичный музыкальный фон",
        "хочу веселую музыку для яркого видео",
        "подбери трек, который звучит позитивно",
        "нужна музыка с ощущением радости",
        "хочу что-то легкое, доброе и позитивное",
        "подбери трек для happy mood",
        "нужна бодрая и светлая музыка",
    ],

    "energetic_electronic": [
        "хочу энергичную электронную музыку",
        "подбери динамичный электронный трек",
        "нужна музыка для активного видео",
        "хочу быстрый электронный трек",
        "подбери музыку с сильной энергией",
        "нужна динамичная музыка с битом",
        "хочу электронную музыку для движения",
        "подбери трек для спортивного ролика",
        "нужна энергичная музыка для action видео",
        "хочу музыку с плотным электронным звучанием",
        "подбери что-то танцевальное и энергичное",
        "нужен трек с драйвом",
        "хочу музыку с быстрым темпом",
        "подбери электронный трек для динамичной сцены",
        "нужна музыка для монтажа с быстрыми кадрами",
        "хочу клубный электронный вайб",
        "подбери энергичный dance трек",
        "нужна музыка, которая добавляет движение",
        "хочу что-то мощное и электронное",
        "подбери трек для активного городского видео",
        "нужна музыка с выраженным ритмом",
        "хочу электронный фон с энергией",
        "подбери трек для бодрого видео",
        "нужна динамичная современная музыка",
        "хочу трек для спортивной нарезки",
        "подбери музыку для быстрого темпа",
        "нужен energetic electronic soundtrack",
        "хочу музыку, которая звучит современно и активно",
        "подбери электронный трек с драйвом",
        "нужна музыка для видео с движением",
    ],

    "calm_relaxing": [
        "хочу спокойную расслабляющую музыку",
        "подбери мягкий фоновый трек",
        "нужна спокойная музыка без резкой динамики",
        "хочу ненавязчивую расслабленную музыку",
        "подбери спокойный инструментальный фон",
        "нужна музыка для медленного видео",
        "хочу мягкую и спокойную композицию",
        "подбери relaxing music",
        "нужна музыка для спокойной атмосферы",
        "хочу плавный фоновый трек",
        "подбери музыку без напряжения",
        "нужна тихая и мягкая музыка",
        "хочу calm ambient background",
        "подбери спокойный трек для природы",
        "нужна музыка для расслабляющего видео",
        "хочу что-то медитативное и мягкое",
        "подбери музыку для peaceful mood",
        "нужна спокойная акустическая музыка",
        "хочу трек без агрессии и драматизма",
        "подбери легкую фоновую музыку",
        "нужна музыка, которая не отвлекает",
        "хочу спокойный chill трек",
        "подбери музыку для тихой сцены",
        "нужен расслабленный музыкальный фон",
        "хочу плавную спокойную музыку",
        "подбери трек для уютного видео",
        "нужна музыка для slow mood",
        "хочу мягкое спокойное звучание",
        "подбери спокойную музыку для монтажа",
        "нужен нейтральный расслабляющий фон",
    ],

    "dark_tense": [
        "хочу напряженную темную музыку",
        "подбери тревожный кинематографичный трек",
        "нужна драматичная музыка",
        "хочу мрачную атмосферную музыку",
        "подбери suspense soundtrack",
        "нужен темный напряженный фон",
        "хочу музыку для тревожной сцены",
        "подбери драматичный cinematic трек",
        "нужна музыка с ощущением опасности",
        "хочу dark cinematic music",
        "подбери трек для хоррор-атмосферы",
        "нужна напряженная музыка для видео",
        "хочу тревожный саундтрек",
        "подбери музыку для mystery mood",
        "нужен мрачный фон без веселья",
        "хочу музыку для страшной или тревожной сцены",
        "подбери dark ambient трек",
        "нужна музыка для драматического момента",
        "хочу что-то темное и кинематографичное",
        "подбери напряженный trailer-like трек",
        "нужна музыка с suspense эффектом",
        "хочу трек с мрачным настроением",
        "подбери музыку для tense video",
        "нужен драматичный темный саундтрек",
        "хочу музыку, которая усиливает тревогу",
        "подбери музыку для ночной мрачной сцены",
        "нужен dark tense background",
        "хочу тревожное атмосферное звучание",
        "подбери трек для напряженного монтажа",
        "нужна кинематографичная темная музыка",
    ],

    "aggressive_intense": [
        "хочу жесткую интенсивную музыку",
        "подбери мощный агрессивный трек",
        "нужна роковая энергичная музыка",
        "хочу heavy rock настроение",
        "подбери агрессивный саундтрек",
        "нужна музыка с сильным напором",
        "хочу интенсивный трек для динамики",
        "подбери мощную музыку для action видео",
        "нужен трек с агрессивным звучанием",
        "хочу что-то тяжелое и энергичное",
        "подбери hard rock или metal вайб",
        "нужна музыка для экстремального ролика",
        "хочу aggressive intense music",
        "подбери трек с мощными гитарами",
        "нужна жесткая музыка для резкого монтажа",
        "хочу музыку с давлением и драйвом",
        "подбери интенсивный роковый фон",
        "нужен мощный трек для спортивного видео",
        "хочу трек с тяжелой энергетикой",
        "подбери музыку для брутальной сцены",
        "нужна сильная и напористая музыка",
        "хочу aggressive soundtrack",
        "подбери музыку для экстремального спорта",
        "нужен тяжелый динамичный трек",
        "хочу интенсивный фон без спокойствия",
        "подбери трек для angry mood",
        "нужна музыка с максимальным драйвом",
        "хочу мощный intense rock трек",
        "подбери жесткую музыку для видео",
        "нужен энергичный агрессивный звук",
    ],

    "dreamy_atmospheric": [
        "хочу атмосферную мечтательную музыку",
        "подбери ambient трек",
        "нужна глубокая атмосферная музыка",
        "хочу dreamy background music",
        "подбери музыку с воздушной атмосферой",
        "нужен трек для мечтательного настроения",
        "хочу пространственный ambient фон",
        "подбери атмосферную музыку для красивого видео",
        "нужна музыка с ощущением глубины",
        "хочу мягкий dreamy soundtrack",
        "подбери трек для эстетичного видео",
        "нужна музыка для атмосферного монтажа",
        "хочу ambient cinematic mood",
        "подбери музыку для видео с природой",
        "нужна мечтательная спокойная музыка",
        "хочу трек с пространственным звучанием",
        "подбери атмосферную instrumental музыку",
        "нужен воздушный фон для визуала",
        "хочу музыку для созерцательной сцены",
        "подбери deep atmospheric трек",
        "нужна музыка с мягкой кинематографичностью",
        "хочу dreamy ambient музыку",
        "подбери трек для красивого slow motion видео",
        "нужен фон с ощущением пространства",
        "хочу музыку для вдохновляющей атмосферы",
        "подбери атмосферный трек без агрессии",
        "нужна музыка для эстетичного визуального контента",
        "хочу легкий ambient фон",
        "подбери трек для dream mood",
        "нужна глубокая и мягкая атмосфера",
    ],

    "sad_melancholic": [
        "хочу грустную меланхоличную музыку",
        "подбери печальный эмоциональный трек",
        "нужна музыка для грустного видео",
        "хочу sad piano mood",
        "подбери melancholic soundtrack",
        "нужна медленная эмоциональная музыка",
        "хочу трек с ощущением грусти",
        "подбери музыку для печальной сцены",
        "нужна меланхоличная атмосфера",
        "хочу спокойную грустную музыку",
        "подбери emotional sad трек",
        "нужна музыка для драматичного момента",
        "хочу трек с ностальгическим настроением",
        "подбери грустный инструментальный фон",
        "нужна музыка с печальным оттенком",
        "хочу melancholic background music",
        "подбери трек для sadness mood",
        "нужна эмоциональная музыка без радости",
        "хочу медленный печальный саундтрек",
        "подбери грустную музыку для видео",
        "нужен трек для трогательной сцены",
        "хочу музыку с чувством потери",
        "подбери тихую меланхоличную композицию",
        "нужен sad cinematic фон",
        "хочу музыку для сентиментального видео",
        "подбери печальный piano track",
        "нужна грустная, но мягкая музыка",
        "хочу трек для melancholic mood",
        "подбери музыку для спокойной грусти",
        "нужна эмоциональная и медленная композиция",
    ],

    "romantic_warm": [
        "хочу теплую романтичную музыку",
        "подбери мягкий романтический трек",
        "нужна музыка для нежного видео",
        "хочу warm romantic mood",
        "подбери музыку с ощущением любви",
        "нужен мягкий эмоциональный фон",
        "хочу теплую акустическую музыку",
        "подбери романтичный instrumental трек",
        "нужна музыка для трогательного момента",
        "хочу gentle romantic music",
        "подбери мягкую музыку для пары",
        "нужна теплая спокойная композиция",
        "хочу музыку с нежной атмосферой",
        "подбери warm acoustic трек",
        "нужен романтический фон без драматизма",
        "хочу приятную теплую музыку",
        "подбери музыку для sentimental mood",
        "нужна мягкая эмоциональная музыка",
        "хочу трек с уютным звучанием",
        "подбери романтическую фоновую музыку",
        "нужен теплый музыкальный фон",
        "хочу музыку для красивого личного момента",
        "подбери нежный piano или guitar трек",
        "нужна музыка с мягким романтическим вайбом",
        "хочу спокойную теплую композицию",
        "подбери трек для love mood",
        "нужна романтичная музыка для видео",
        "хочу теплую инструментальную музыку",
        "подбери gentle warm soundtrack",
        "нужна нежная музыка без резкости",
    ],

    "neutral_background": [
        "хочу нейтральную фоновую музыку",
        "подбери ненавязчивый background трек",
        "нужна музыка, которая не отвлекает",
        "хочу простой фоновый саундтрек",
        "подбери нейтральную музыку для монтажа",
        "нужен спокойный универсальный фон",
        "хочу музыку для презентационного видео",
        "подбери corporate background music",
        "нужна универсальная инструментальная музыка",
        "хочу фон без яркой эмоции",
        "подбери минималистичную музыку",
        "нужна музыка для нейтрального ролика",
        "хочу легкий background без акцентов",
        "подбери трек для документального видео",
        "нужна фоновая музыка без вокала",
        "хочу нейтральный инструментальный фон",
        "подбери ненавязчивую музыку для контента",
        "нужен простой музыкальный фон",
        "хочу музыку для спокойного объясняющего видео",
        "подбери background track",
        "нужна музыка, не задающая сильное настроение",
        "хочу универсальный фон для видео",
        "подбери музыку для tutorial видео",
        "нужен нейтральный саундтрек",
        "хочу фоновую музыку без драматизма",
        "подбери минимальный спокойный трек",
        "нужна музыка для обычного видео",
        "хочу unobtrusive background music",
        "подбери трек для no emotion видео",
        "нужен фоновый instrumental track",
    ],

    "playful_fun": [
        "хочу игривую забавную музыку",
        "подбери смешной легкий трек",
        "нужна музыка для веселого забавного видео",
        "хочу playful music",
        "подбери quirky трек",
        "нужна музыка для комедийной сцены",
        "хочу детскую или игровую атмосферу",
        "подбери забавный soundtrack",
        "нужен легкий игривый фон",
        "хочу музыку для funny mood",
        "подбери трек для смешного ролика",
        "нужна музыка с комедийным оттенком",
        "хочу cartoon-like music",
        "подбери веселую необычную музыку",
        "нужен игривый фон для видео",
        "хочу музыку для шутливой сцены",
        "подбери playful background",
        "нужна легкая забавная музыка",
        "хочу quirky funny soundtrack",
        "подбери трек для детского видео",
        "нужна музыка с игровой атмосферой",
        "хочу смешной и легкий музыкальный фон",
        "подбери музыку для забавного монтажа",
        "нужен playful upbeat трек",
        "хочу музыку с toy/cartoon настроением",
        "подбери трек для comedy mood",
        "нужна веселая, но не агрессивная музыка",
        "хочу легкий шутливый трек",
        "подбери музыку для funny video",
        "нужна игривая позитивная композиция",
    ],
}


# =========================
# CATEGORY SIMILARITY
# =========================

# query_audio_score показывает, насколько пользовательский запрос
# соответствует категории конкретного трека.
#
# 1.0 — прямое совпадение
# 0.6–0.8 — близкие категории
# 0.2–0.4 — слабое соответствие
# 0.0 — не соответствует
CATEGORY_SIMILARITY: Dict[str, Dict[str, float]] = {
    "happy_upbeat": {
        "happy_upbeat": 1.0,
        "playful_fun": 0.8,
        "energetic_electronic": 0.6,
        "romantic_warm": 0.5,
        "calm_relaxing": 0.3,
        "neutral_background": 0.2,
        "dreamy_atmospheric": 0.2,
        "sad_melancholic": 0.0,
        "dark_tense": 0.0,
        "aggressive_intense": 0.1,
    },

    "energetic_electronic": {
        "energetic_electronic": 1.0,
        "happy_upbeat": 0.6,
        "aggressive_intense": 0.6,
        "playful_fun": 0.5,
        "dark_tense": 0.3,
        "dreamy_atmospheric": 0.2,
        "neutral_background": 0.1,
        "calm_relaxing": 0.0,
        "sad_melancholic": 0.0,
        "romantic_warm": 0.0,
    },

    "calm_relaxing": {
        "calm_relaxing": 1.0,
        "neutral_background": 0.8,
        "dreamy_atmospheric": 0.7,
        "romantic_warm": 0.6,
        "sad_melancholic": 0.5,
        "happy_upbeat": 0.3,
        "playful_fun": 0.2,
        "dark_tense": 0.1,
        "energetic_electronic": 0.0,
        "aggressive_intense": 0.0,
    },

    "dark_tense": {
        "dark_tense": 1.0,
        "aggressive_intense": 0.7,
        "dreamy_atmospheric": 0.5,
        "sad_melancholic": 0.4,
        "neutral_background": 0.2,
        "energetic_electronic": 0.2,
        "calm_relaxing": 0.0,
        "happy_upbeat": 0.0,
        "romantic_warm": 0.0,
        "playful_fun": 0.0,
    },

    "aggressive_intense": {
        "aggressive_intense": 1.0,
        "dark_tense": 0.7,
        "energetic_electronic": 0.6,
        "sad_melancholic": 0.2,
        "dreamy_atmospheric": 0.1,
        "neutral_background": 0.0,
        "happy_upbeat": 0.0,
        "calm_relaxing": 0.0,
        "romantic_warm": 0.0,
        "playful_fun": 0.0,
    },

    "dreamy_atmospheric": {
        "dreamy_atmospheric": 1.0,
        "calm_relaxing": 0.7,
        "dark_tense": 0.5,
        "romantic_warm": 0.5,
        "neutral_background": 0.4,
        "sad_melancholic": 0.4,
        "happy_upbeat": 0.2,
        "energetic_electronic": 0.2,
        "aggressive_intense": 0.1,
        "playful_fun": 0.1,
    },

    "sad_melancholic": {
        "sad_melancholic": 1.0,
        "calm_relaxing": 0.6,
        "dreamy_atmospheric": 0.6,
        "romantic_warm": 0.5,
        "dark_tense": 0.4,
        "neutral_background": 0.2,
        "happy_upbeat": 0.0,
        "energetic_electronic": 0.0,
        "aggressive_intense": 0.0,
        "playful_fun": 0.0,
    },

    "romantic_warm": {
        "romantic_warm": 1.0,
        "calm_relaxing": 0.7,
        "dreamy_atmospheric": 0.5,
        "happy_upbeat": 0.5,
        "sad_melancholic": 0.4,
        "neutral_background": 0.3,
        "playful_fun": 0.2,
        "energetic_electronic": 0.0,
        "dark_tense": 0.0,
        "aggressive_intense": 0.0,
    },

    "neutral_background": {
        "neutral_background": 1.0,
        "calm_relaxing": 0.8,
        "dreamy_atmospheric": 0.5,
        "romantic_warm": 0.4,
        "happy_upbeat": 0.3,
        "sad_melancholic": 0.3,
        "playful_fun": 0.2,
        "energetic_electronic": 0.1,
        "dark_tense": 0.1,
        "aggressive_intense": 0.0,
    },

    "playful_fun": {
        "playful_fun": 1.0,
        "happy_upbeat": 0.8,
        "energetic_electronic": 0.5,
        "neutral_background": 0.2,
        "romantic_warm": 0.2,
        "calm_relaxing": 0.1,
        "dreamy_atmospheric": 0.1,
        "sad_melancholic": 0.0,
        "dark_tense": 0.0,
        "aggressive_intense": 0.0,
    },
}


# =========================
# UTILS
# =========================

def read_csv_safely(path: Path) -> pd.DataFrame:
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError as e:
            last_error = e

    raise last_error


def normalize_category(value: str) -> str:
    value = str(value).strip().lower()
    value = value.replace("-", "_")
    value = re.sub(r"\s+", "_", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")


def get_query_audio_score(query_category: str, track_category: str) -> float:
    query_category = normalize_category(query_category)
    track_category = normalize_category(track_category)

    if query_category == track_category:
        return 1.0

    return float(
        CATEGORY_SIMILARITY
        .get(query_category, {})
        .get(track_category, 0.0)
    )


def build_query_id(video_id: str, query_category: str, template_id: int) -> str:
    return f"{video_id}__{query_category}__tpl_{template_id}"


# =========================
# AUGMENTATION LOGIC
# =========================

def validate_input_df(df: pd.DataFrame, split_name: str) -> None:
    required = {
        "video_id",
        "video_title",
        "video_path",
        "duration_s",
        "a_priori_emotion",
        "track_id",
        "audio_path",
        "audio_category",
        "compatibility_score",
        "target_binary",
        "split",
    }

    missing = required - set(df.columns)

    if missing:
        raise ValueError(f"В {split_name} не хватает столбцов: {missing}")

    if df.empty:
        raise ValueError(f"{split_name} пустой.")

    if df["video_id"].isna().any():
        raise ValueError(f"В {split_name} есть пустые video_id.")

    if df["track_id"].isna().any():
        raise ValueError(f"В {split_name} есть пустые track_id.")

    if df["audio_category"].isna().any():
        raise ValueError(f"В {split_name} есть пустые audio_category.")

    if df["compatibility_score"].isna().any():
        raise ValueError(f"В {split_name} есть пустые compatibility_score.")


def choose_query_categories(df: pd.DataFrame) -> List[str]:
    existing_categories = sorted(
        df["audio_category"]
        .dropna()
        .astype(str)
        .map(normalize_category)
        .unique()
        .tolist()
    )

    categories_with_templates = [
        category for category in existing_categories
        if category in QUERY_TEMPLATES and len(QUERY_TEMPLATES[category]) > 0
    ]

    if not categories_with_templates:
        raise ValueError(
            "Не найдено ни одной audio_category, для которой есть QUERY_TEMPLATES."
        )

    return categories_with_templates


def sample_templates_for_category(
    category: str,
    n_templates: int,
    rng: random.Random,
) -> List[tuple]:
    templates = QUERY_TEMPLATES[category]

    if len(templates) <= n_templates:
        sampled_indices = list(range(len(templates)))
    else:
        sampled_indices = rng.sample(range(len(templates)), n_templates)

    return [(idx, templates[idx]) for idx in sampled_indices]


def augment_split_with_queries(
    df: pd.DataFrame,
    split_name: str,
    templates_per_category: int,
    random_state: int,
) -> pd.DataFrame:
    validate_input_df(df, split_name)

    rng = random.Random(random_state)

    df = df.copy()
    df["audio_category"] = df["audio_category"].map(normalize_category)
    df["video_audio_score"] = pd.to_numeric(
        df["compatibility_score"],
        errors="coerce",
    )

    # Сохраняем старую бинарную метку, которая была рассчитана только по video-audio.
    df["target_binary_video_only"] = pd.to_numeric(
        df["target_binary"],
        errors="coerce",
    ).fillna(0).astype(int)

    query_categories = choose_query_categories(df)

    augmented_rows = []

    # Важно: генерируем запросы на уровне video_id,
    # а потом применяем каждый запрос ко всем трекам данного видео.
    # Это имитирует реальный inference:
    # один user_query + одно video -> scores по всем tracks.
    for video_id, video_group in df.groupby("video_id", sort=False):
        video_group = video_group.copy()

        for query_category in query_categories:
            sampled_templates = sample_templates_for_category(
                category=query_category,
                n_templates=templates_per_category,
                rng=rng,
            )

            for template_id, user_query in sampled_templates:
                query_id = build_query_id(
                    video_id=str(video_id),
                    query_category=query_category,
                    template_id=template_id,
                )

                temp = video_group.copy()

                temp["query_id"] = query_id
                temp["user_query"] = user_query
                temp["query_audio_category"] = query_category
                temp["query_template_id"] = template_id

                temp["query_audio_score"] = temp["audio_category"].apply(
                    lambda track_category: get_query_audio_score(
                        query_category=query_category,
                        track_category=track_category,
                    )
                )

                temp["final_compatibility_score"] = (
                    VIDEO_SCORE_WEIGHT * temp["video_audio_score"]
                    + QUERY_SCORE_WEIGHT * temp["query_audio_score"]
                )

                temp["target_binary"] = (
                    temp["final_compatibility_score"] >= POSITIVE_THRESHOLD
                ).astype(int)

                augmented_rows.append(temp)

    if not augmented_rows:
        raise ValueError(f"Не удалось сформировать строки для {split_name}.")

    out = pd.concat(augmented_rows, ignore_index=True)

    # Удобный порядок колонок.
    preferred_cols = [
        "split",
        "query_id",
        "user_query",
        "query_audio_category",
        "query_template_id",

        "video_id",
        "video_title",
        "video_path",
        "duration_s",
        "a_priori_emotion",

        "track_id",
        "audio_path",
        "selected_genre",
        "audio_category",

        "video_audio_score",
        "query_audio_score",
        "final_compatibility_score",

        "target_binary",
        "target_binary_video_only",
    ]

    existing_preferred = [c for c in preferred_cols if c in out.columns]
    other_cols = [c for c in out.columns if c not in existing_preferred]

    out = out[existing_preferred + other_cols].copy()

    return out


# =========================
# STATS
# =========================

def print_stats(df: pd.DataFrame, split_name: str) -> None:
    print(f"\n=== {split_name} with queries ===")
    print(f"Rows: {len(df)}")
    print(f"Unique videos: {df['video_id'].nunique()}")
    print(f"Unique tracks: {df['track_id'].nunique()}")
    print(f"Unique user queries: {df['query_id'].nunique()}")
    print(f"Positive ratio: {df['target_binary'].mean():.3f}")

    print("\nQuery categories:")
    print(df[["query_id", "query_audio_category"]].drop_duplicates()["query_audio_category"].value_counts())

    print("\nTrack audio categories:")
    print(df[["track_id", "audio_category"]].drop_duplicates()["audio_category"].value_counts())

    print("\nPositive ratio by query category:")
    print(
        df.groupby("query_audio_category")["target_binary"]
        .agg(["count", "sum", "mean"])
        .rename(columns={"sum": "positive_count", "mean": "positive_ratio"})
        .sort_values("count", ascending=False)
    )

    print("\nPositive ratio by video emotion:")
    print(
        df.groupby("a_priori_emotion")["target_binary"]
        .agg(["count", "sum", "mean"])
        .rename(columns={"sum": "positive_count", "mean": "positive_ratio"})
        .sort_values("count", ascending=False)
    )


def save_query_template_table() -> None:
    rows = []

    for category, templates in QUERY_TEMPLATES.items():
        for template_id, template in enumerate(templates):
            rows.append({
                "query_audio_category": category,
                "query_template_id": template_id,
                "user_query": template,
            })

    pd.DataFrame(rows).to_csv(
        OUT_DIR / "query_templates.csv",
        index=False,
        encoding="utf-8-sig",
    )


def save_category_similarity_table() -> None:
    rows = []

    for query_category, scores in CATEGORY_SIMILARITY.items():
        for track_category, score in scores.items():
            rows.append({
                "query_audio_category": query_category,
                "track_audio_category": track_category,
                "query_audio_score": score,
            })

    pd.DataFrame(rows).to_csv(
        OUT_DIR / "category_similarity.csv",
        index=False,
        encoding="utf-8-sig",
    )


# =========================
# MAIN
# =========================

def process_file(
    input_path: Path,
    output_path: Path,
    split_name: str,
    templates_per_category: int,
    seed_offset: int,
) -> pd.DataFrame:
    if not input_path.exists():
        raise FileNotFoundError(f"Не найден файл: {input_path}")

    df = read_csv_safely(input_path)
    df.columns = [str(c).strip() for c in df.columns]

    augmented = augment_split_with_queries(
        df=df,
        split_name=split_name,
        templates_per_category=templates_per_category,
        random_state=RANDOM_STATE + seed_offset,
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    augmented.to_csv(output_path, index=False, encoding="utf-8-sig")

    print_stats(augmented, split_name)
    print(f"\nSaved: {output_path}")

    return augmented


def main() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    train = process_file(
        input_path=TRAIN_IN,
        output_path=TRAIN_OUT,
        split_name="train",
        templates_per_category=TRAIN_TEMPLATES_PER_CATEGORY,
        seed_offset=0,
    )

    val = process_file(
        input_path=VAL_IN,
        output_path=VAL_OUT,
        split_name="val",
        templates_per_category=VAL_TEMPLATES_PER_CATEGORY,
        seed_offset=1000,
    )

    test = process_file(
        input_path=TEST_IN,
        output_path=TEST_OUT,
        split_name="test",
        templates_per_category=TEST_TEMPLATES_PER_CATEGORY,
        seed_offset=2000,
    )

    save_query_template_table()
    save_category_similarity_table()

    print("\nDone.")
    print(f"Output directory: {OUT_DIR}")
    print(f"Train: {TRAIN_OUT}")
    print(f"Val:   {VAL_OUT}")
    print(f"Test:  {TEST_OUT}")
    print(f"Query templates: {OUT_DIR / 'query_templates.csv'}")
    print(f"Category similarity: {OUT_DIR / 'category_similarity.csv'}")


if __name__ == "__main__":
    main()


=== train with queries ===
Rows: 97920
Unique videos: 68
Unique tracks: 60
Unique user queries: 1632
Positive ratio: 0.281

Query categories:
query_audio_category
aggressive_intense      204
calm_relaxing           204
dark_tense              204
dreamy_atmospheric      204
energetic_electronic    204
happy_upbeat            204
romantic_warm           204
sad_melancholic         204
Name: count, dtype: int64

Track audio categories:
audio_category
calm_relaxing           20
dreamy_atmospheric      12
dark_tense               8
aggressive_intense       7
happy_upbeat             6
sad_melancholic          3
energetic_electronic     3
romantic_warm            1
Name: count, dtype: int64

Positive ratio by query category:
                      count  positive_count  positive_ratio
query_audio_category                                       
aggressive_intense    12240            1896        0.154902
calm_relaxing         12240            5310        0.433824
dark_tense            12240  

## EDA of dataset user text query + video --> audio track

In [14]:
import re
from pathlib import Path
from typing import Dict, List, Optional

import matplotlib.pyplot as plt
import pandas as pd


# =========================
# PATHS
# =========================

DATA_DIR = Path("final_dataset_with_queries")

TRAIN_CSV = DATA_DIR / "train_with_queries.csv"
VAL_CSV = DATA_DIR / "val_with_queries.csv"
TEST_CSV = DATA_DIR / "test_with_queries.csv"

OUT_DIR = Path("eda_report_with_queries")
FIG_DIR = OUT_DIR / "figures"
TABLE_DIR = OUT_DIR / "tables"

QUERY_TEMPLATES_CSV = DATA_DIR / "query_templates.csv"
CATEGORY_SIMILARITY_CSV = DATA_DIR / "category_similarity.csv"


# =========================
# CONFIG
# =========================

EXPECTED_COLUMNS = [
    "split",
    "query_id",
    "user_query",
    "query_audio_category",
    "query_template_id",

    "video_id",
    "video_title",
    "video_path",
    "duration_s",
    "a_priori_emotion",

    "track_id",
    "audio_path",
    "selected_genre",
    "audio_category",

    "video_audio_score",
    "query_audio_score",
    "final_compatibility_score",

    "target_binary",
    "target_binary_video_only",
]

MIN_AUDIO_CATEGORY_TRACKS = 3
MIN_VIDEO_EMOTION_COUNT = 3

RUSSIAN_TOKEN_RE = re.compile(r"\w+", flags=re.UNICODE)


# =========================
# UTILS
# =========================

def ensure_dirs() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    TABLE_DIR.mkdir(parents=True, exist_ok=True)


def read_csv_safely(path: Path) -> pd.DataFrame:
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
    last_error = None

    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError as e:
            last_error = e

    raise last_error


def file_exists(path_value) -> bool:
    if pd.isna(path_value):
        return False

    path_value = str(path_value).strip()

    if not path_value:
        return False

    path = Path(path_value)
    return path.exists() and path.is_file() and path.stat().st_size > 0


def count_words(text: str) -> int:
    if pd.isna(text):
        return 0

    return len(RUSSIAN_TOKEN_RE.findall(str(text)))


def save_bar(
    series: pd.Series,
    title: str,
    xlabel: str,
    ylabel: str,
    out_path: Path,
    rotation: int = 45,
) -> None:
    plt.figure(figsize=(10, 5))
    series.plot(kind="bar")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=rotation, ha="right")
    plt.tight_layout()
    plt.savefig(out_path, dpi=220)
    plt.close()


def save_hist(
    values: pd.Series,
    title: str,
    xlabel: str,
    ylabel: str,
    out_path: Path,
    bins: int = 30,
) -> None:
    values = pd.to_numeric(values, errors="coerce").dropna()

    plt.figure(figsize=(8, 5))
    plt.hist(values, bins=bins)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(out_path, dpi=220)
    plt.close()


def save_heatmap(
    table: pd.DataFrame,
    title: str,
    out_path: Path,
    colorbar_label: str = "value",
) -> None:
    table = table.fillna(0)

    plt.figure(figsize=(12, 7))
    plt.imshow(table.values, aspect="auto")
    plt.colorbar(label=colorbar_label)
    plt.title(title)
    plt.xticks(range(len(table.columns)), table.columns, rotation=45, ha="right")
    plt.yticks(range(len(table.index)), table.index)
    plt.tight_layout()
    plt.savefig(out_path, dpi=220)
    plt.close()


# =========================
# LOAD DATA
# =========================

def load_split(path: Path, split_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Не найден файл: {path}")

    df = read_csv_safely(path)
    df.columns = [str(c).strip() for c in df.columns]

    missing = set(EXPECTED_COLUMNS) - set(df.columns)
    if missing:
        raise ValueError(f"В {path} не хватает столбцов: {missing}")

    df = df.copy()
    df["split"] = split_name

    return df


def load_all_data() -> pd.DataFrame:
    train = load_split(TRAIN_CSV, "train")
    val = load_split(VAL_CSV, "val")
    test = load_split(TEST_CSV, "test")

    df = pd.concat([train, val, test], ignore_index=True)

    numeric_cols = [
        "duration_s",
        "video_audio_score",
        "query_audio_score",
        "final_compatibility_score",
        "target_binary",
        "target_binary_video_only",
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["target_binary"] = df["target_binary"].fillna(0).astype(int)
    df["target_binary_video_only"] = df["target_binary_video_only"].fillna(0).astype(int)

    df["user_query"] = df["user_query"].astype(str)
    df["user_query_len_chars"] = df["user_query"].str.len()
    df["user_query_len_words"] = df["user_query"].apply(count_words)

    return df


def get_unique_videos(df: pd.DataFrame) -> pd.DataFrame:
    return df[
        [
            "split",
            "video_id",
            "video_title",
            "video_path",
            "duration_s",
            "a_priori_emotion",
        ]
    ].drop_duplicates(subset=["split", "video_id"]).copy()


def get_unique_tracks(df: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "track_id",
        "audio_path",
        "selected_genre",
        "audio_category",
    ]

    existing = [c for c in cols if c in df.columns]

    return df[existing].drop_duplicates(subset=["track_id"]).copy()


def get_unique_queries(df: pd.DataFrame) -> pd.DataFrame:
    return df[
        [
            "split",
            "query_id",
            "user_query",
            "query_audio_category",
            "query_template_id",
            "video_id",
        ]
    ].drop_duplicates(subset=["split", "query_id"]).copy()


# =========================
# QUALITY CHECKS
# =========================

def run_quality_checks(df: pd.DataFrame) -> pd.DataFrame:
    checks = []

    def add(name: str, value, status: str, comment: str = ""):
        checks.append({
            "check": name,
            "value": value,
            "status": status,
            "comment": comment,
        })

    add("rows_total", len(df), "info")
    add("unique_videos_total", df["video_id"].nunique(), "info")
    add("unique_tracks_total", df["track_id"].nunique(), "info")
    add("unique_query_ids_total", df["query_id"].nunique(), "info")
    add("unique_user_query_texts_total", df["user_query"].nunique(), "info")

    missing_values = df[EXPECTED_COLUMNS].isna().sum()
    total_missing = int(missing_values.sum())

    add(
        "missing_values_in_expected_columns",
        total_missing,
        "ok" if total_missing == 0 else "warning",
        "Если есть пропуски, см. missing_values.csv.",
    )

    duplicate_triplets = df.duplicated(
        subset=["split", "query_id", "video_id", "track_id"]
    ).sum()

    add(
        "duplicate_query_video_track_triplets",
        int(duplicate_triplets),
        "ok" if duplicate_triplets == 0 else "warning",
        "Дубликаты одной и той же тройки query-video-track нежелательны.",
    )

    positive_ratio = df["target_binary"].mean()

    add(
        "overall_positive_ratio",
        round(float(positive_ratio), 4),
        "ok" if 0.15 <= positive_ratio <= 0.5 else "warning",
        "Для retrieval/ranking допустима умеренная доля positive; слишком высокий баланс делает задачу менее строгой.",
    )

    # Leakage: одно и то же видео не должно быть в нескольких split.
    video_split_counts = df.groupby("video_id")["split"].nunique()
    video_leakage = int((video_split_counts > 1).sum())

    add(
        "video_id_split_leakage",
        video_leakage,
        "ok" if video_leakage == 0 else "error",
        "Одно и то же видео не должно попадать в разные split.",
    )

    # Для треков leakage допустим, потому что словарь треков общий.
    track_split_counts = df.groupby("track_id")["split"].nunique()
    tracks_in_all_splits = int((track_split_counts == 3).sum())

    add(
        "tracks_present_in_all_splits",
        tracks_in_all_splits,
        "info",
        "Это нормально: треки являются фиксированным словарём рекомендаций.",
    )

    # Повтор текстовых шаблонов в split не является ошибкой, но это ограничение.
    query_text_split_counts = df.groupby("user_query")["split"].nunique()
    query_text_overlap = int((query_text_split_counts > 1).sum())

    add(
        "user_query_text_overlap_between_splits",
        query_text_overlap,
        "info",
        "Повтор шаблонов допустим, если проверяется обобщение на новые видео; для проверки новых формулировок нужен отдельный split по шаблонам.",
    )

    tracks = get_unique_tracks(df)

    audio_category_counts = tracks["audio_category"].value_counts()
    rare_audio_categories = audio_category_counts[
        audio_category_counts < MIN_AUDIO_CATEGORY_TRACKS
    ].index.tolist()

    add(
        "rare_audio_categories",
        ", ".join(rare_audio_categories) if rare_audio_categories else "none",
        "ok" if not rare_audio_categories else "warning",
        f"Категории с числом треков меньше {MIN_AUDIO_CATEGORY_TRACKS}.",
    )

    videos = get_unique_videos(df)
    emotion_counts = videos["a_priori_emotion"].value_counts()
    rare_emotions = emotion_counts[
        emotion_counts < MIN_VIDEO_EMOTION_COUNT
    ].index.tolist()

    add(
        "rare_video_emotions",
        ", ".join(rare_emotions) if rare_emotions else "none",
        "ok" if not rare_emotions else "warning",
        f"Эмоции с числом видео меньше {MIN_VIDEO_EMOTION_COUNT}.",
    )

    if "video_path" in df.columns:
        unique_video_paths = videos["video_path"].dropna().drop_duplicates()
        missing_video_files = int((~unique_video_paths.apply(file_exists)).sum())

        add(
            "missing_video_files",
            missing_video_files,
            "ok" if missing_video_files == 0 else "warning",
            "Часть video_path не существует или файл пустой.",
        )

    if "audio_path" in df.columns:
        unique_audio_paths = tracks["audio_path"].dropna().drop_duplicates()
        missing_audio_files = int((~unique_audio_paths.apply(file_exists)).sum())

        add(
            "missing_audio_files",
            missing_audio_files,
            "ok" if missing_audio_files == 0 else "warning",
            "Часть audio_path не существует или файл пустой.",
        )

    checks_df = pd.DataFrame(checks)

    checks_df.to_csv(TABLE_DIR / "quality_checks.csv", index=False, encoding="utf-8-sig")
    missing_values.to_csv(TABLE_DIR / "missing_values.csv", encoding="utf-8-sig")

    return checks_df


# =========================
# SAVE TABLES
# =========================

def save_tables(df: pd.DataFrame) -> None:
    videos = get_unique_videos(df)
    tracks = get_unique_tracks(df)
    queries = get_unique_queries(df)

    # Basic distributions
    df.groupby("split").size().rename("rows").to_csv(
        TABLE_DIR / "rows_by_split.csv",
        encoding="utf-8-sig",
    )

    videos.groupby("split")["video_id"].nunique().rename("unique_videos").to_csv(
        TABLE_DIR / "unique_videos_by_split.csv",
        encoding="utf-8-sig",
    )

    queries.groupby("split")["query_id"].nunique().rename("unique_queries").to_csv(
        TABLE_DIR / "unique_queries_by_split.csv",
        encoding="utf-8-sig",
    )

    videos["a_priori_emotion"].value_counts().to_csv(
        TABLE_DIR / "video_emotion_distribution.csv",
        encoding="utf-8-sig",
    )

    tracks["audio_category"].value_counts().to_csv(
        TABLE_DIR / "track_audio_category_distribution.csv",
        encoding="utf-8-sig",
    )

    queries["query_audio_category"].value_counts().to_csv(
        TABLE_DIR / "query_audio_category_distribution.csv",
        encoding="utf-8-sig",
    )

    df["target_binary"].value_counts().sort_index().to_csv(
        TABLE_DIR / "target_distribution.csv",
        encoding="utf-8-sig",
    )

    # Positive ratios
    df.groupby("split")["target_binary"].agg(["count", "sum", "mean"]).rename(
        columns={"sum": "positive_count", "mean": "positive_ratio"}
    ).to_csv(
        TABLE_DIR / "positive_ratio_by_split.csv",
        encoding="utf-8-sig",
    )

    df.groupby("query_audio_category")["target_binary"].agg(["count", "sum", "mean"]).rename(
        columns={"sum": "positive_count", "mean": "positive_ratio"}
    ).sort_values("count", ascending=False).to_csv(
        TABLE_DIR / "positive_ratio_by_query_category.csv",
        encoding="utf-8-sig",
    )

    df.groupby("a_priori_emotion")["target_binary"].agg(["count", "sum", "mean"]).rename(
        columns={"sum": "positive_count", "mean": "positive_ratio"}
    ).sort_values("count", ascending=False).to_csv(
        TABLE_DIR / "positive_ratio_by_video_emotion.csv",
        encoding="utf-8-sig",
    )

    df.groupby("audio_category")["target_binary"].agg(["count", "sum", "mean"]).rename(
        columns={"sum": "positive_count", "mean": "positive_ratio"}
    ).sort_values("count", ascending=False).to_csv(
        TABLE_DIR / "positive_ratio_by_track_audio_category.csv",
        encoding="utf-8-sig",
    )

    # Matrices
    query_track_matrix = df.pivot_table(
        index="query_audio_category",
        columns="audio_category",
        values="query_audio_score",
        aggfunc="mean",
    )

    query_track_matrix.to_csv(
        TABLE_DIR / "mean_query_audio_score_query_x_track_category.csv",
        encoding="utf-8-sig",
    )

    final_score_matrix = df.pivot_table(
        index="query_audio_category",
        columns="audio_category",
        values="final_compatibility_score",
        aggfunc="mean",
    )

    final_score_matrix.to_csv(
        TABLE_DIR / "mean_final_score_query_x_track_category.csv",
        encoding="utf-8-sig",
    )

    positive_matrix_query_track = df.pivot_table(
        index="query_audio_category",
        columns="audio_category",
        values="target_binary",
        aggfunc="mean",
    )

    positive_matrix_query_track.to_csv(
        TABLE_DIR / "positive_ratio_query_x_track_category.csv",
        encoding="utf-8-sig",
    )

    emotion_query_matrix = df.pivot_table(
        index="a_priori_emotion",
        columns="query_audio_category",
        values="target_binary",
        aggfunc="mean",
    )

    emotion_query_matrix.to_csv(
        TABLE_DIR / "positive_ratio_emotion_x_query_category.csv",
        encoding="utf-8-sig",
    )

    # Query length
    queries[["split", "query_id", "user_query", "query_audio_category"]].copy().assign(
        user_query_len_chars=queries["user_query"].str.len(),
        user_query_len_words=queries["user_query"].apply(count_words),
    ).to_csv(
        TABLE_DIR / "unique_queries_with_lengths.csv",
        index=False,
        encoding="utf-8-sig",
    )


# =========================
# SAVE FIGURES
# =========================

def save_figures(df: pd.DataFrame) -> None:
    videos = get_unique_videos(df)
    tracks = get_unique_tracks(df)
    queries = get_unique_queries(df)

    # 1. Распределение эмоций видео
    save_bar(
        videos["a_priori_emotion"].value_counts(),
        title="Distribution of video emotions",
        xlabel="Video emotion",
        ylabel="Number of unique videos",
        out_path=FIG_DIR / "video_emotion_distribution.png",
    )

    # 2. Распределение категорий треков
    save_bar(
        tracks["audio_category"].value_counts(),
        title="Distribution of track audio categories",
        xlabel="Track audio category",
        ylabel="Number of unique tracks",
        out_path=FIG_DIR / "track_audio_category_distribution.png",
    )

    # 3. Распределение query-категорий
    save_bar(
        queries["query_audio_category"].value_counts(),
        title="Distribution of query categories",
        xlabel="Query audio category",
        ylabel="Number of unique user queries",
        out_path=FIG_DIR / "query_audio_category_distribution.png",
    )

    # 4. Target distribution
    save_bar(
        df["target_binary"].value_counts().sort_index(),
        title="Target distribution with user queries",
        xlabel="Target binary",
        ylabel="Number of query-video-track rows",
        out_path=FIG_DIR / "target_distribution.png",
        rotation=0,
    )

    # 5. Positive ratio by split
    save_bar(
        df.groupby("split")["target_binary"].mean().sort_index(),
        title="Positive ratio by split",
        xlabel="Split",
        ylabel="Positive ratio",
        out_path=FIG_DIR / "positive_ratio_by_split.png",
        rotation=0,
    )

    # 6. Positive ratio by query category
    save_bar(
        df.groupby("query_audio_category")["target_binary"].mean().sort_values(ascending=False),
        title="Positive ratio by query category",
        xlabel="Query audio category",
        ylabel="Positive ratio",
        out_path=FIG_DIR / "positive_ratio_by_query_category.png",
    )

    # 7. Positive ratio by video emotion
    save_bar(
        df.groupby("a_priori_emotion")["target_binary"].mean().sort_values(ascending=False),
        title="Positive ratio by video emotion",
        xlabel="Video emotion",
        ylabel="Positive ratio",
        out_path=FIG_DIR / "positive_ratio_by_video_emotion.png",
    )

    # 8. Длины запросов
    save_hist(
        queries["user_query"].apply(count_words),
        title="User query length distribution",
        xlabel="Number of words",
        ylabel="Number of unique queries",
        out_path=FIG_DIR / "user_query_length_words.png",
        bins=15,
    )

    # 9. Score distributions
    save_hist(
        df["video_audio_score"],
        title="Video-audio score distribution",
        xlabel="video_audio_score",
        ylabel="Number of rows",
        out_path=FIG_DIR / "video_audio_score_distribution.png",
        bins=20,
    )

    save_hist(
        df["query_audio_score"],
        title="Query-audio score distribution",
        xlabel="query_audio_score",
        ylabel="Number of rows",
        out_path=FIG_DIR / "query_audio_score_distribution.png",
        bins=20,
    )

    save_hist(
        df["final_compatibility_score"],
        title="Final compatibility score distribution",
        xlabel="final_compatibility_score",
        ylabel="Number of rows",
        out_path=FIG_DIR / "final_compatibility_score_distribution.png",
        bins=20,
    )

    # 10. Heatmaps
    positive_query_track = df.pivot_table(
        index="query_audio_category",
        columns="audio_category",
        values="target_binary",
        aggfunc="mean",
    )

    save_heatmap(
        positive_query_track,
        title="Positive ratio: query category x track category",
        out_path=FIG_DIR / "heatmap_positive_ratio_query_x_track.png",
        colorbar_label="positive ratio",
    )

    final_score_query_track = df.pivot_table(
        index="query_audio_category",
        columns="audio_category",
        values="final_compatibility_score",
        aggfunc="mean",
    )

    save_heatmap(
        final_score_query_track,
        title="Mean final score: query category x track category",
        out_path=FIG_DIR / "heatmap_final_score_query_x_track.png",
        colorbar_label="mean final score",
    )

    emotion_query = df.pivot_table(
        index="a_priori_emotion",
        columns="query_audio_category",
        values="target_binary",
        aggfunc="mean",
    )

    save_heatmap(
        emotion_query,
        title="Positive ratio: video emotion x query category",
        out_path=FIG_DIR / "heatmap_positive_ratio_emotion_x_query.png",
        colorbar_label="positive ratio",
    )


# =========================
# TEXT SUMMARY
# =========================

def save_text_summary(df: pd.DataFrame, checks: pd.DataFrame) -> None:
    videos = get_unique_videos(df)
    tracks = get_unique_tracks(df)
    queries = get_unique_queries(df)

    path = OUT_DIR / "eda_summary.txt"

    lines = []

    lines.append("EDA SUMMARY FOR USER-QUERY DATASET")
    lines.append("=" * 80)
    lines.append("")

    lines.append("Dataset size:")
    lines.append(f"- rows: {len(df)}")
    lines.append(f"- unique videos: {df['video_id'].nunique()}")
    lines.append(f"- unique tracks: {df['track_id'].nunique()}")
    lines.append(f"- unique query ids: {df['query_id'].nunique()}")
    lines.append(f"- unique query texts: {df['user_query'].nunique()}")
    lines.append(f"- positive ratio: {df['target_binary'].mean():.3f}")
    lines.append("")

    lines.append("Rows by split:")
    lines.append(str(df["split"].value_counts()))
    lines.append("")

    lines.append("Unique videos by split:")
    lines.append(str(videos.groupby("split")["video_id"].nunique()))
    lines.append("")

    lines.append("Unique queries by split:")
    lines.append(str(queries.groupby("split")["query_id"].nunique()))
    lines.append("")

    lines.append("Video emotion distribution:")
    lines.append(str(videos["a_priori_emotion"].value_counts()))
    lines.append("")

    lines.append("Track audio category distribution:")
    lines.append(str(tracks["audio_category"].value_counts()))
    lines.append("")

    lines.append("Query category distribution:")
    lines.append(str(queries["query_audio_category"].value_counts()))
    lines.append("")

    lines.append("Positive ratio by split:")
    lines.append(str(df.groupby("split")["target_binary"].mean()))
    lines.append("")

    lines.append("Positive ratio by query category:")
    lines.append(str(df.groupby("query_audio_category")["target_binary"].mean().sort_values(ascending=False)))
    lines.append("")

    lines.append("Potential issues:")
    warnings = checks[checks["status"].isin(["warning", "error"])]

    if warnings.empty:
        lines.append("- No major issues detected.")
    else:
        for _, row in warnings.iterrows():
            lines.append(f"- [{row['status']}] {row['check']}: {row['value']} — {row['comment']}")

    lines.append("")
    lines.append("Interpretation:")
    lines.append(
        "- The dataset now represents triples: user_query + video + track -> target."
    )
    lines.append(
        "- Query categories are generated synthetically and balanced by construction."
    )
    lines.append(
        "- Track categories remain imbalanced because they depend on the selected MTG-Jamendo subset."
    )
    lines.append(
        "- Split leakage is checked by video_id; tracks are shared across splits because the track dictionary is fixed."
    )

    path.write_text("\n".join(lines), encoding="utf-8")


# =========================
# MAIN
# =========================

def main() -> None:
    ensure_dirs()

    df = load_all_data()

    checks = run_quality_checks(df)
    save_tables(df)
    save_figures(df)
    save_text_summary(df, checks)

    print("\nEDA finished.")
    print(f"Report directory: {OUT_DIR}")
    print(f"Figures: {FIG_DIR}")
    print(f"Tables: {TABLE_DIR}")

    print("\nMain stats:")
    print(f"Rows: {len(df)}")
    print(f"Unique videos: {df['video_id'].nunique()}")
    print(f"Unique tracks: {df['track_id'].nunique()}")
    print(f"Unique query ids: {df['query_id'].nunique()}")
    print(f"Unique query texts: {df['user_query'].nunique()}")
    print(f"Positive ratio: {df['target_binary'].mean():.3f}")

    print("\nQuality checks:")
    print(checks)


if __name__ == "__main__":
    main()


EDA finished.
Report directory: eda_report_with_queries
Figures: eda_report_with_queries\figures
Tables: eda_report_with_queries\tables

Main stats:
Rows: 126720
Unique videos: 98
Unique tracks: 60
Unique query ids: 2112
Unique query texts: 240
Positive ratio: 0.285

Quality checks:
                                     check          value   status  \
0                               rows_total         126720     info   
1                      unique_videos_total             98     info   
2                      unique_tracks_total             60     info   
3                   unique_query_ids_total           2112     info   
4            unique_user_query_texts_total            240     info   
5       missing_values_in_expected_columns              0       ok   
6     duplicate_query_video_track_triplets              0       ok   
7                   overall_positive_ratio         0.2848       ok   
8                   video_id_split_leakage              0       ok   
9             t